# Concentration_function_real_400_CPU_lepkosc_v17

**Wersja przyspieszona.** Jedyna zmiana w obliczeniach: `p_weierstrass_matrix` liczy ℘ wektorowo w `pfast.py` (ta sama formuła co wcześniej). Plik `pfast.py` musi leżeć obok klatek (w Colabie: wgrać razem z `klatka_*.png`). Działa na GPU (CuPy) i bez GPU (NumPy). Szczegóły: `README.md`.

**Wersja zgodna z `Import-e2-CORRECTED.nb`.** W poprawionym Mathematica: `E2(0) = π`, `e2 = (1/le²)·ΣΣ℘ + π`, `e22 = (1/le³)·Σ_k |Σ_m E2|²`, `a1 = x + i·y`.

Zmiany względem poprzedniej wersji notebooka tylko w `prepare()` (komórka 3b):
1. `a = x + i·y` (było `y + i·x`),
2. `g2 = 189.0727` (było zaokrąglone `189.073`).

`M_e2 = 1/N²` i diagonala `E2 = π` pozostają jak były. Stare wyniki w komórkach 4 i 5 usunięto – uruchom je ponownie.

**Zmiany w wersji 16 względem v15:**
1. Zakomentowano zapis wykresów 3D (e22, Re_e2, Im_e2, abs_e2) w `zapisz_wizualizacje` (komórka 3a) — kod pozostawiony do ewentualnego użycia.
2. Dodano komórkę 8: losowe bakterie w oknie 100×100 px (przypadki A: N=39 i B: N=43) — obliczenie f, Re A3 i μ_e/μ tą samą procedurą co dla okien z rzeczywistych klatek; osie rysunku unormowane do [0, 1].

**Zmiany w wersji 17 względem v16:**
1. Nowa komórka 3e (wklejona po 3d): współczynniki **A4, A5, A6** z wzorów (41)–(43) pracy Czapla, Drygaś, Gluzman, Ligocki, Mityushev, *Materials* 2025, 18, 5041 (funkcje wygenerowane automatycznie z `elasticity2.tex`). Sumy strukturalne dowolnego rzędu liczone algorytmem łańcuchowym (iloczyn macierz–wektor), funkcje E_n^{(1)}, n = 4, 5, 6 wyprowadzone z (A7) przez różniczkowanie i sprawdzone z bezpośrednimi sumami sieciowymi (`_selftest_A456()` uruchamia się przy wykonaniu komórki).
2. `lepkosc_okna` zwraca dodatkowo: `Re_A4, Im_A4, Re_A5, Im_A5, Re_A6, Im_A6` oraz `mu_e_A4, mu_e_A5, mu_e_A6` (μ_e/μ = (1+s)/(1−s), s = f + ReA3 f³ + ReA4 f⁴ + …). Dotychczasowe wartości (e33, e4, A3, mu_e_pade) nie zmieniają się. Przełączniki: `LICZ_A456`, `RHO1..3`, `KAPPA`, `S2_E2`, `DIAG_E31` (komórka 3e).
3. `przetworz_klatke` (3b) zapisuje dodatkowo `real{i}_lepkosc_okna.txt` (TXT) obok `real{i}_lepkosc_okna.csv` (nowe kolumny w CSV).
4. Komórka 9 (raport lepkości): kolumny A4–A6 i mu_e_A4–A6 w statystykach klatek.
5. Nowa komórka 5b (po komórce 5): szeregi czasowe — `result_A456_okna_szeregi.csv` (każde okno po klatkach), `result_A456_klatki_szereg.csv` i `.txt` (statystyki klatek po kolejnych i).

Kolejność uruchamiania: 1, 2, 3a, 3b, 3c, 3d, **3e**, 4 (pętla), 5, **5b**, …

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib.util

# Dodaj Drive do sys.path (żeby Python szukał tam modułów)
if '/content/drive/MyDrive' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive')

# 1. Czy plik istnieje?
sciezka = '/content/drive/MyDrive/pfast.py'
print(f"{sciezka}:", "✅" if os.path.isfile(sciezka) else "❌ brak")

# 2. Czy Python go widzi?
spec = importlib.util.find_spec('pfast')
print("Moduł widoczny:", spec.origin if spec else "❌ NIE")

# 3. Czy import działa?
try:
    from pfast import cp, p_weierstrass_matrix_fast
    print("Import OK ✅")
except ImportError as e:
    print(f"Import błąd ❌: {e}")

In [ ]:
# Komórka 1: importy i parametry globalne

import os
import sys
import time
import math
import re
import csv
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
!pip install -q cupy-cuda12x scikit-image opencv-python-headless mpmath

# --- Wymuś UTF-8 dla stdout/stderr ---
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')
os.environ['PYTHONIOENCODING'] = 'utf-8'

# =====================================================================
# ŚCIEŻKI I ZAKRES OBRAZKÓW
# =====================================================================
path = '/content/drive/MyDrive/'
i_min, i_max = 1, 239    # wszystkie klatki

# =====================================================================
# PODKATALOGI NA PLIKI WYNIKOWE
# =====================================================================
PATH_TXT = path + 'TXT/'
PATH_PNG = path + 'PNG/'
PATH_CSV = path + 'CSV/'

os.makedirs(PATH_TXT, exist_ok=True)
os.makedirs(PATH_PNG, exist_ok=True)
os.makedirs(PATH_CSV, exist_ok=True)

# =====================================================================
# ORYGINALNY OBRAZ (klatka_N.png) I OBCIĘCIE CZARNYCH PASKÓW
# =====================================================================
y_w_orig, x_w_orig = 652, 640
Y_CUT_TOP = 6
Y_CUT_BOT = 8

y_w = y_w_orig - Y_CUT_TOP - Y_CUT_BOT    # = 638
x_w = x_w_orig                             # = 640

# =====================================================================
# ROI (OKNO PŁYWAJĄCE) — 220 × 220 px
# =====================================================================
y_d, x_d = 100, 100

# =====================================================================
# PARAMETRY CZYSZCZENIA I KWANTYZACJI
# =====================================================================
r_min = 8
r_min_detekcji = 2
r_max_detekcji = 8
DIST_SCALANIE = 10

# =====================================================================
# DELTA — ZAPAS PASKÓW DLA TORUSA
# =====================================================================
delta = 3 * r_min         # = 24

# =====================================================================
# ROZMIAR TORUSA
# =====================================================================
y_w_torus = y_w + 2 * delta    # = 686
x_w_torus = x_w + 2 * delta    # = 688

# =====================================================================
# OKNA PŁYWAJĄCE
# =====================================================================
LICZBA_OKIEN_Y = 50
LICZBA_OKIEN_X = 50

y_0_min = 0
x_0_min = 0
Y_0_RANGE = y_w - y_d
X_0_RANGE = x_w - x_d

krok_y = Y_0_RANGE // max(1, LICZBA_OKIEN_Y - 1)
krok_x = X_0_RANGE // max(1, LICZBA_OKIEN_X - 1)

y_0_max = y_0_min + (LICZBA_OKIEN_Y - 1) * krok_y
x_0_max = x_0_min + (LICZBA_OKIEN_X - 1) * krok_x

nr_min, nr_max = 0, LICZBA_OKIEN_Y - 1
mr_min, mr_max = 0, LICZBA_OKIEN_X - 1

assert y_0_max + y_d <= y_w
assert x_0_max + x_d <= x_w

# =====================================================================
# PARAMETRY DETEKTORÓW
# =====================================================================
BLOB_MIN_SIGMA = 2
BLOB_MAX_SIGMA = 6
BLOB_NUM_SIGMA = 5
BLOB_THRESHOLD = 0.1

# =====================================================================
# PNG / TXT / CSV
# =====================================================================
SAVE_PNG_FOR_I = 1
PNG_PREFIX = 'real'
TXT_PREFIX = 'real'
SAVE_KLATKA_BEZ_PASKOW = True
SAVE_KLATKA_TORUS = True

# =====================================================================
# DEBUG
# =====================================================================
DEBUG_NR = nr_max
DEBUG_MR = mr_max

# =====================================================================
# LIMITY BEZPIECZEŃSTWA
# =====================================================================
MAX_PUNKTOW_E2 = 500
BLOCK_SIZE_E2 = 256

# =====================================================================
# WIZUALIZACJE
# =====================================================================
VIEW_ANGLES = [(30, 45), (60, 120), (90, 180), (15, 270)]
COLUMNS_CSV = ["N", "y", "x", "Re(e2)", "Im(e2)", "|e2|", "e22"]
WIELKOSCI = [
    ("e22",    "e22",    "e22"),
    ("Re(e2)", "Re_e2",  "Re(e2)"),
    ("Im(e2)", "Im_e2",  "Im(e2)"),
    ("|e2|",   "abs_e2", "|e2|"),
]
GRID_N = 100
WIELKOSCI_STAT = ["Re(e2)", "Im(e2)", "|e2|", "e22"]

# =====================================================================
# FUNKCJA POMOCNICZA — diagnostyka pamięci GPU
# =====================================================================
def gpu_mem_info(tag=""):
    try:
        import cupy as cp
        pool = cp.get_default_memory_pool()
        used = pool.used_bytes() / 1e9
        total = pool.total_bytes() / 1e9
        print(f"[GPU {tag}] used={used:.2f} GB, total={total:.2f} GB")
    except Exception as e:
        print(f"[GPU {tag}] brak info: {e}")

print(f"=== PARAMETRY STARTOWE ===")
print(f"Obrazy: klatka_{{i:03d}}.png, i ∈ [{i_min}, {i_max}]")
print(f"Oryginalny obraz:  {y_w_orig} × {x_w_orig} (y × x)")
print(f"Obcięcie:          {Y_CUT_TOP} px z góry, {Y_CUT_BOT} px z dołu")
print(f"Po obcięciu:       {y_w} × {x_w} (y × x)")
print(f"Delta:             {delta} px (= 3 × r_min = 3 × {r_min})")
print(f"Torus:             {y_w_torus} × {x_w_torus} (y × x)")
print(f"ROI (okno):        {y_d} × {x_d} (y × x)")
print(f"Liczba okien:      {LICZBA_OKIEN_Y} × {LICZBA_OKIEN_X} = "
      f"{LICZBA_OKIEN_Y * LICZBA_OKIEN_X}")
print(f"Zakres y_0:        [{y_0_min}, {y_0_max}], krok = {krok_y}")
print(f"Zakres x_0:        [{x_0_min}, {x_0_max}], krok = {krok_x}")
print(f"Detektory:         sigma ∈ [{BLOB_MIN_SIGMA}, {BLOB_MAX_SIGMA}], "
      f"threshold = {BLOB_THRESHOLD}")
print(f"Katalogi wynikowe:")
print(f"  TXT: {PATH_TXT}")
print(f"  PNG: {PATH_PNG}")
print(f"  CSV: {PATH_CSV}")
print(f"==========================")

In [ ]:
# Komórka 2: funkcje pomocnicze

def check_kdtree(array, label=""):
    """Diagnostyka: znajduje dwa najbliższe punkty i wypisuje ich odległość."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 2:
        return len(array)
    tree = cKDTree(array[:, :2])
    d, idx = tree.query(array[:, :2], k=2)
    min_dist = d[:, 1].min()
    argmin = int(d[:, 1].argmin())
    print(f"[{label}] Najbliższe: {array[argmin, :2]} i {array[idx[argmin, 1], :2]}")
    print(f"[{label}] Odległość: {min_dist:.4f}, N={len(array)}")
    return len(array)


def clean_kdtree(array, dist, label=""):
    """Scala punkty bliższe niż `dist` w jeden punkt (płaska metryka)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 2:
        return array
    tree = cKDTree(array[:, :2])
    pairs = tree.query_pairs(r=dist, output_type='ndarray')
    if len(pairs) == 0:
        print(f"[{label}] Brak par < {dist}. N={len(array)}")
        return array

    parent = np.arange(len(array))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for a, b in pairs:
        union(int(a), int(b))

    from collections import defaultdict
    klastry = defaultdict(list)
    for idx in range(len(array)):
        klastry[find(idx)].append(idx)

    wynik = np.array([array[l].mean(axis=0) for l in klastry.values()])
    print(f"[{label}] clean_kdtree: {len(array)} → {len(wynik)}")
    return wynik


def clean_kdtree_torus(array, y_0, x_0, y_d, x_d, dist, label=""):
    """Scalanie punktów z METRYKĄ TORUSOWĄ (9 kopii + etykiety)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 2:
        return array

    shifts = [
        (0, 0), (0, +x_d), (0, -x_d),
        (+y_d, 0), (-y_d, 0),
        (+y_d, +x_d), (+y_d, -x_d),
        (-y_d, +x_d), (-y_d, -x_d),
    ]
    kopie, etykiety = [], []
    for k, (dy, dx) in enumerate(shifts):
        kopia = array[:, :2].copy()
        kopia[:, 0] += dy
        kopia[:, 1] += dx
        kopia[:, 0] = y_0 + (kopia[:, 0] - y_0) % y_d
        kopia[:, 1] = x_0 + (kopia[:, 1] - x_0) % x_d
        kopie.append(kopia)
        etykiety.append(np.full(len(kopia), k, dtype=np.int32))
    kopie = np.concatenate(kopie)
    etykiety = np.concatenate(etykiety)

    tree = cKDTree(kopie)
    pairs = tree.query_pairs(r=dist, output_type='ndarray')
    if len(pairs) == 0:
        print(f"[{label}] Brak par < {dist} (torus). N={len(array)}")
        return array

    parent = np.arange(len(kopie))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    for a, b in pairs:
        union(int(a), int(b))

    from collections import defaultdict
    klastry = defaultdict(list)
    for idx in range(len(kopie)):
        klastry[find(idx)].append(idx)

    wynik = []
    for idx_lista in klastry.values():
        oryginaly = [i for i in idx_lista if etykiety[i] == 0]
        if len(oryginaly) > 0:
            sr = kopie[oryginaly].mean(axis=0)
            wynik.append(sr)

    wynik = np.array(wynik)
    print(f"[{label}] clean_kdtree_torus: {len(array)} → {len(wynik)}")
    return wynik


def kwantyzuj_do_siatki(array, y_0, x_0, y_d, x_d, krok):
    """Kwantyzacja punktów do regularnej siatki o kroku `krok` (px)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 1:
        return np.zeros((0, 2))
    i_idx = np.floor((array[:, 0] - y_0) / krok).astype(np.int64)
    j_idx = np.floor((array[:, 1] - x_0) / krok).astype(np.int64)

    from collections import defaultdict
    grupy = defaultdict(list)
    for k in range(len(array)):
        grupy[(i_idx[k], j_idx[k])].append(k)

    wynik = []
    for (i, j), idx_lista in grupy.items():
        sr = array[idx_lista].mean(axis=0)
        wynik.append(sr)

    wynik = np.array(wynik)
    print(f"[kwantyzuj] krok={krok:.3f}: {len(array)} → {len(wynik)}")
    return wynik


def replikuj_9x(array, y_d, x_d):
    """9 kopii punktów (bez zawijania)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) == 0:
        return np.zeros((0, 2)), np.zeros(0, dtype=np.int32)
    shifts = [
        (0, 0), (0, +x_d), (0, -x_d),
        (+y_d, 0), (-y_d, 0),
        (+y_d, +x_d), (+y_d, -x_d),
        (-y_d, +x_d), (-y_d, -x_d),
    ]
    kopie, etykiety = [], []
    for k, (dy, dx) in enumerate(shifts):
        kopia = array[:, :2].copy()
        kopia[:, 0] += dy
        kopia[:, 1] += dx
        kopie.append(kopia)
        etykiety.append(np.full(len(kopia), k, dtype=np.int32))
    return np.concatenate(kopie), np.concatenate(etykiety)


def symetryzuj(blobs_9x, y_0, x_0, y_d, x_d, dist=None):
    """Filtracja punktów z ROI + opcjonalna kwantyzacja (płaska metryka)."""
    blobs_9x = np.asarray(blobs_9x, dtype=np.float64)
    if len(blobs_9x) == 0:
        return np.zeros((0, 2))
    mask_roi = (
        (blobs_9x[:, 0] >= y_0) & (blobs_9x[:, 0] < y_0 + y_d) &
        (blobs_9x[:, 1] >= x_0) & (blobs_9x[:, 1] < x_0 + x_d)
    )
    roi = blobs_9x[mask_roi][:, :2].copy()
    if len(roi) == 0:
        return np.zeros((0, 2))
    if dist is not None and dist > 0 and len(roi) > 1:
        roi = kwantyzuj_do_siatki(roi, y_0, x_0, y_d, x_d, krok=dist)
    roi[:, 0] -= y_0
    roi[:, 1] -= x_0
    return roi

In [ ]:
# Komórka 3a: I/O + wizualizacje + statystyki

import re

RE_LINE = re.compile(
    r'N=(\d+),\s*okno:\s*(\d+),(\d+),\s*e2=\((.+?)\),\s*e22=([\d.eE+-]+)'
)
RE_IMAG = re.compile(r'([-+])\s*([\d.]+(?:[eE][-+]?\d+)?)j$')


def _zapisz_wspolrzedne(blobs_roi, i, nr, mr, png_prefix='real'):
    """Zapisuje współrzędne punktów (Z NORMALIZACJĄ do [0, 1]) do TXT i CSV."""
    import numpy as np

    N = len(blobs_roi)
    txt_file = PATH_TXT + f'{png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.txt'
    csv_file = PATH_CSV + f'{png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.csv'

    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write(f"# Współrzędne punktów (Z NORMALIZACJĄ do [0, 1]) dla "
                f"i={i}, nr={nr}, mr={mr}, N={N}\n")
        f.write(f"# Układ: lokalny ROI, znormalizowany przez y_d={y_d}, x_d={x_d}\n")
        f.write(f"# Liczba punktów: {N}\n")
        f.write(f"# Format: x_norm  y_norm   (obie w [0, 1])\n")
        f.write("#" + "-" * 40 + "\n")
        for p in blobs_roi:
            yb, xb = p[0], p[1]
            x_norm = xb / x_d
            y_norm = yb / y_d
            f.write(f"{x_norm:>12.6f}  {y_norm:>12.6f}\n")

    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['x_norm', 'y_norm'])
        for p in blobs_roi:
            yb, xb = p[0], p[1]
            x_norm = xb / x_d
            y_norm = yb / y_d
            writer.writerow([f"{x_norm:.6f}", f"{y_norm:.6f}"])


def _zapisz_png(image, blobs_roi, i, nr, mr, png_prefix='real'):
    """Zapisuje PNG 1:1 (220×220) + TXT + CSV per okno."""
    import os
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np

    N = len(blobs_roi)
    fname = PATH_PNG + f'{png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.png'
    if os.path.exists(fname):
        print(f"[WIZ] UWAGA: {fname} już istnieje — nadpisuję!")

    y_max, x_max = image.shape[0], image.shape[1]

    DPI = 80
    fig = plt.figure(figsize=(x_max / DPI, y_max / DPI), dpi=DPI)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()

    ax.imshow(image, cmap='gray', origin='upper',
              extent=[0, x_max, y_max, 0],
              interpolation='nearest',
              aspect='auto')

    if len(blobs_roi) > 0:
        mask = (
            (blobs_roi[:, 0] >= 0) & (blobs_roi[:, 0] < y_max) &
            (blobs_roi[:, 1] >= 0) & (blobs_roi[:, 1] < x_max)
        )
        blobs_ok = blobs_roi[mask]
        n_poza = len(blobs_roi) - len(blobs_ok)
        if n_poza > 0:
            print(f"[WIZ] i={i} nr={nr} mr={mr}: {n_poza} punktów poza ROI — obcinam.")
        ax.scatter(blobs_ok[:, 1], blobs_ok[:, 0],
                   s=8, c='red', marker='o', linewidths=0, zorder=3)

    ax.set_xlim(0, x_max)
    ax.set_ylim(y_max, 0)

    try:
        fig.savefig(fname, dpi=DPI)
        plt.close(fig)
        print(f"[WIZ] Zapisano: {fname}  ({x_max}×{y_max} px, 1:1)")
    except Exception as e:
        print(f"[WIZ] Błąd zapisu PNG i={i} nr={nr} mr={mr}: {e}")
        try:
            plt.close('all')
        except Exception:
            pass

    try:
        _zapisz_wspolrzedne(blobs_roi, i, nr, mr, png_prefix)
        print(f"[WSP] Zapisano: {png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.txt i .csv "
              f"(N={N}, kolumny x_norm, y_norm)")
    except Exception as e:
        print(f"[WSP] Błąd zapisu współrzędnych i={i} nr={nr} mr={mr}: {e}")


def konwertuj_txt_na_csv(txt_file, csv_file):
    """Konwertuje realN.txt → realN.csv."""
    if not os.path.exists(txt_file):
        return None

    n_wierszy = 0
    with open(txt_file, 'r', encoding='utf-8') as f_in, \
         open(csv_file, 'w', newline='', encoding='utf-8') as f_out:
        writer = csv.writer(f_out)
        writer.writerow(COLUMNS_CSV)

        for line in f_in:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            m = RE_LINE.match(line)
            if not m:
                continue

            N, y, x, e2_raw, e22 = m.groups()
            e2_clean = e2_raw.strip()
            match_c = RE_IMAG.search(e2_clean)
            if not match_c:
                continue
            sign = match_c.group(1)
            im_part = match_c.group(2)
            re_part = e2_clean[:match_c.start()].strip()
            im_value = ('-' if sign == '-' else '') + im_part

            try:
                re_float = float(re_part)
                im_float = float(im_value)
                e22_float = float(e22)
            except ValueError:
                continue

            modul = math.sqrt(re_float**2 + im_float**2)
            writer.writerow([N, y, x, re_part, im_value, modul, e22])
            n_wierszy += 1

    return n_wierszy


def wycinek_torus(img_torus, y_0, x_0, y_d, x_d):
    """Wycinek z torusa z ZAWIJANIEM modulo (H_t, W_t)."""
    import numpy as np
    H_t, W_t = img_torus.shape
    y_idx = (np.arange(y_0, y_0 + y_d) % H_t).astype(np.int64)
    x_idx = (np.arange(x_0, x_0 + x_d) % W_t).astype(np.int64)
    roi = img_torus[np.ix_(y_idx, x_idx)]
    return roi


def _wycinek_z_uzupelnieniem(img, y_0, x_0, y_d, x_d, fill_value=0):
    """Wycinek z obrazu `img` z uzupełnieniem czernią poza granicami."""
    import numpy as np
    H, W = img.shape
    roi = np.full((y_d, x_d), fill_value, dtype=img.dtype)

    y_src_start = max(0, y_0)
    y_src_end = min(H, y_0 + y_d)
    x_src_start = max(0, x_0)
    x_src_end = min(W, x_0 + x_d)

    if y_src_end <= y_src_start or x_src_end <= x_src_start:
        return roi

    y_dst_start = y_src_start - y_0
    y_dst_end = y_src_end - y_0
    x_dst_start = x_src_start - x_0
    x_dst_end = x_src_end - x_0

    roi[y_dst_start:y_dst_end, x_dst_start:x_dst_end] = \
        img[y_src_start:y_src_end, x_src_start:x_src_end]

    return roi


def _plot_surface_with_contours(df, z_col, z_label):
    """Rysuje figurę 2×2 z 4 wykresami 3D."""
    x = df["x"].to_numpy()
    y = df["y"].to_numpy()
    z = df[z_col].to_numpy()

    X_grid, Y_grid = np.meshgrid(
        np.linspace(min(x), max(x), GRID_N),
        np.linspace(min(y), max(y), GRID_N)
    )
    Z_grid = griddata((x, y), z, (X_grid, Y_grid), method='linear')

    fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                             subplot_kw={'projection': '3d'})

    for ax, (elev, azim) in zip(axes.ravel(), VIEW_ANGLES):
        ax.scatter(x, y, z, c=z, cmap="viridis", marker="o", label="Dane")
        ax.plot_surface(X_grid, Y_grid, Z_grid, cmap="viridis", alpha=0.8)
        ax.contour3D(X_grid, Y_grid, Z_grid, 50, cmap='viridis')
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel(z_label)

        x_ticks = np.arange(min(x), max(x) + 1, 20)
        y_ticks = np.arange(min(y), max(y) + 1, 20)
        ax.set_xticks(x_ticks)
        ax.set_yticks(y_ticks)
        ax.set_xticklabels([str(int(i)) if i % 40 == 0 else "" for i in x_ticks])
        ax.set_yticklabels([str(int(i)) if i % 40 == 0 else "" for i in y_ticks])

        ax.view_init(elev, azim)
        ax.set_title(f"{z_label}  |  View: Elev={elev}°, Azim={azim}°")

    fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05,
                        wspace=0.2, hspace=0.3)
    return fig


def _plot_contour_2d(df, z_col, z_label):
    """Rysuje figurę 2D z konturami."""
    x = df["x"].to_numpy()
    y = df["y"].to_numpy()
    z = df[z_col].to_numpy()

    X_grid, Y_grid = np.meshgrid(
        np.linspace(min(x), max(x), GRID_N),
        np.linspace(min(y), max(y), GRID_N)
    )
    Z_grid = griddata((x, y), z, (X_grid, Y_grid), method='linear')

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    contourf = ax.contourf(X_grid, Y_grid, Z_grid, levels=30, cmap='viridis')
    contour = ax.contour(X_grid, Y_grid, Z_grid, levels=15,
                         colors='black', linewidths=0.5, alpha=0.6)
    ax.clabel(contour, inline=True, fontsize=7, fmt='%.2g')
    ax.scatter(x, y, c='red', s=8, marker='o', linewidths=0,
               zorder=3, label='Dane')

    cbar = fig.colorbar(contourf, ax=ax, shrink=0.8)
    cbar.set_label(z_label)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"ContourPlot: {z_label}")
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    return fig


def _wczytaj_csv(csv_file):
    """Wczytuje realN.csv do DataFrame (do wizualizacji)."""
    if not os.path.exists(csv_file):
        return None
    try:
        df = pd.read_csv(csv_file, delimiter=",", decimal=".",
                         comment="#", header=0, encoding='utf-8')
        if len(df.columns) == len(COLUMNS_CSV):
            df.columns = COLUMNS_CSV
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        potrzebne = ["x", "y", "e22", "Re(e2)", "Im(e2)", "|e2|"]
        for c in potrzebne:
            if c not in df.columns:
                return None
        df = df.dropna(subset=potrzebne)
        if len(df) < 4:
            return None
        return df
    except Exception as e:
        print(f"[WARN] Błąd wczytywania {csv_file}: {e}")
        return None


def zapisz_wizualizacje(csv_file, i):
    """Zapisuje 4×3D + 4×2D PNG do PATH_PNG."""
    df = _wczytaj_csv(csv_file)
    if df is None:
        return 0, 0, []

    n_3d, n_2d = 0, 0
    bledy = []
    for z_col, suffix, z_label in WIELKOSCI:
        # [WYŁĄCZONE] zapis wykresów 3D (e22, Re_e2, Im_e2, abs_e2) — zakomentowane, może być użyte później:
        # png_3d = PATH_PNG + f'real{i}_3D_{suffix}.png'
        # try:
        #     fig = _plot_surface_with_contours(df, z_col, z_label)
        #     fig.savefig(png_3d, dpi=100)
        #     plt.close(fig)
        #     n_3d += 1
        # except Exception as e:
        #     bledy.append((i, f"3D {z_label}", str(e)))
        #     try:
        #         plt.close('all')
        #     except Exception:
        #         pass

        png_2d = PATH_PNG + f'real{i}_2D_{suffix}.png'
        try:
            fig = _plot_contour_2d(df, z_col, z_label)
            fig.savefig(png_2d, dpi=100)
            plt.close(fig)
            n_2d += 1
        except Exception as e:
            bledy.append((i, f"2D {z_label}", str(e)))
            try:
                plt.close('all')
            except Exception:
                pass

    return n_3d, n_2d, bledy


def wczytaj_csv_stat(csv_file):
    """Wczytuje realN.csv do statystyk."""
    if not os.path.exists(csv_file):
        return None
    try:
        df = pd.read_csv(csv_file, delimiter=",", decimal=".",
                         comment="#", header=0, encoding='utf-8')
        if len(df.columns) == len(COLUMNS_CSV):
            df.columns = COLUMNS_CSV
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        potrzebne = ["x", "y"] + WIELKOSCI_STAT
        for c in potrzebne:
            if c not in df.columns:
                print(f"[WARN] {csv_file}: brak kolumny {c}")
                return None
        df = df.dropna(subset=potrzebne)
        if len(df) < 1:
            return None
        return df
    except Exception as e:
        print(f"[WARN] Błąd wczytywania {csv_file}: {e}")
        return None


def statystyki(df):
    """Statystyki: mean, std, median, percentyle, IQR."""
    wynik = {}
    for c in WIELKOSCI_STAT:
        v = df[c].to_numpy(dtype=np.float64)
        wynik[c] = {
            'mean':      float(np.mean(v)),
            'std':       float(np.std(v, ddof=0)),
            'std_ddof1': float(np.std(v, ddof=1)) if len(v) > 1 else 0.0,
            'min':       float(np.min(v)),
            'max':       float(np.max(v)),
            'median':    float(np.median(v)),
            'q05':       float(np.percentile(v, 5)),
            'q25':       float(np.percentile(v, 25)),
            'q75':       float(np.percentile(v, 75)),
            'q95':       float(np.percentile(v, 95)),
            'iqr':       float(np.percentile(v, 75) - np.percentile(v, 25)),
            'n':         int(len(v)),
        }
    return wynik


def zapisz_resultN(result_file, i, stats, n_wierszy, y_0_zakres, x_0_zakres):
    """Zapisuje statystyki do resultN.txt."""
    with open(result_file, 'w', encoding='utf-8') as f:
        f.write(f"# Statystyki dla obrazka i={i}\n")
        f.write(f"# Źródło: real{i}.csv\n")
        f.write(f"# Liczba wierszy (okien): {n_wierszy}\n")
        f.write(f"# Zakres okien: y_0 ∈ {y_0_zakres}, x_0 ∈ {x_0_zakres}\n")
        f.write(f"# Format: wielkosc mean std std_ddof1 min max median "
                f"q05 q25 q75 q95 iqr n\n")
        f.write("#" + "-" * 130 + "\n")
        for c in WIELKOSCI_STAT:
            s = stats[c]
            f.write(
                f"{c:>8s}  "
                f"mean={s['mean']:> .6e}  "
                f"std={s['std']:> .6e}  "
                f"std_ddof1={s['std_ddof1']:> .6e}  "
                f"min={s['min']:> .6e}  "
                f"max={s['max']:> .6e}  "
                f"median={s['median']:> .6e}  "
                f"q05={s['q05']:> .6e}  "
                f"q25={s['q25']:> .6e}  "
                f"q75={s['q75']:> .6e}  "
                f"q95={s['q95']:> .6e}  "
                f"iqr={s['iqr']:> .6e}  "
                f"n={s['n']}\n"
            )


def zapisz_statystyki_dla_klatki(i):
    """Wrapper: wczytuje realN.csv, zapisuje resultN.txt."""
    csv_file = PATH_CSV + f'real{i}.csv'
    result_file = PATH_TXT + f'result{i}.txt'

    df = wczytaj_csv_stat(csv_file)
    if df is None:
        return False

    try:
        stats = statystyki(df)
        n_wierszy = len(df)
        try:
            y_0_zakres = f"[{y_0_min}, {y_0_max}], krok={krok_y}"
            x_0_zakres = f"[{x_0_min}, {x_0_max}], krok={krok_x}"
        except NameError:
            y_0_zakres = "(nieznany)"
            x_0_zakres = "(nieznany)"
        zapisz_resultN(result_file, i, stats, n_wierszy, y_0_zakres, x_0_zakres)
        print(f"[STAT] {csv_file} → {result_file}  "
              f"(N={n_wierszy}, e22: mean={stats['e22']['mean']:.4e}, "
              f"median={stats['e22']['median']:.4e})")
        return True
    except Exception as e:
        print(f"[STAT BŁĄD] {csv_file}: {e}")
        return False

In [ ]:
# Komórka 3b: nowy potok torusowy
# =====================================================================
# ZMIANA: zamiast symetryzuj_torus (uśrednianie — tworzy sztuczne punkty
# w środku okna), używamy usun_duplikaty_torus (usuwa jeden z pary —
# ten z WIĘKSZYMI współrzędnymi). Zostaje punkt bliżej (0, 0).
# =====================================================================

import time


_torus_cache = {}
_blobs_cache = {}


# =====================================================================
# PRZYGOTOWANIE TORUSA
# =====================================================================

def przygotuj_torus(i):
    """Wczytuje klatka_N.png, odcina czarne paski, dokleja delta z 4 stron + rogi."""
    import cv2
    import numpy as np

    if i in _torus_cache:
        return _torus_cache[i]

    data_path = path + f'klatka_{i:03d}.png'
    img = cv2.imread(data_path, 0)
    if img is None:
        raise FileNotFoundError(f"Nie znaleziono pliku: {data_path}")

    img_bez_paskow = img[Y_CUT_TOP:y_w_orig - Y_CUT_BOT, :]

    if SAVE_KLATKA_BEZ_PASKOW:
        out_path = path + f'klatka_{i:03d}-bez_paskow.png'
        cv2.imwrite(out_path, img_bez_paskow)
        print(f"[TORUS] Zapisano: {out_path}  "
              f"({img_bez_paskow.shape[0]}×{img_bez_paskow.shape[1]} px)")

    H, W = img_bez_paskow.shape
    img_torus = np.zeros((H + 2 * delta, W + 2 * delta), dtype=img_bez_paskow.dtype)

    img_torus[delta:delta + H, delta:delta + W] = img_bez_paskow
    img_torus[delta:delta + H, delta + W:delta + W + delta] = img_bez_paskow[:, 0:delta]
    img_torus[delta:delta + H, 0:delta] = img_bez_paskow[:, W - delta:W]
    img_torus[delta + H:delta + H + delta, delta:delta + W] = img_bez_paskow[0:delta, :]
    img_torus[0:delta, delta:delta + W] = img_bez_paskow[H - delta:H, :]
    img_torus[0:delta, 0:delta] = img_bez_paskow[H - delta:H, W - delta:W]
    img_torus[0:delta, delta + W:delta + W + delta] = img_bez_paskow[H - delta:H, 0:delta]
    img_torus[delta + H:delta + H + delta, 0:delta] = img_bez_paskow[0:delta, W - delta:W]
    img_torus[delta + H:delta + H + delta, delta + W:delta + W + delta] = \
        img_bez_paskow[0:delta, 0:delta]

    if SAVE_KLATKA_TORUS:
        out_path = path + f'klatka_{i:03d}-torus.png'
        cv2.imwrite(out_path, img_torus)
        print(f"[TORUS] Zapisano: {out_path}  "
              f"({img_torus.shape[0]}×{img_torus.shape[1]} px)")

    _torus_cache[i] = (img_torus, img_bez_paskow)
    return img_torus, img_bez_paskow


# =====================================================================
# DETEKCJA BLOBÓW RAZ NA TORUSIE
# =====================================================================

def wykryj_bloby_na_torusie(i, debug=False):
    """Detekcja bakterii RAZ na torusie + odrzucenie pasków + clean torus."""
    import cv2
    import numpy as np
    from skimage.feature import blob_log, blob_dog
    from math import sqrt

    if i in _blobs_cache:
        return _blobs_cache[i]

    img_torus, img_bez_paskow = przygotuj_torus(i)
    H, W = img_bez_paskow.shape

    def denoise(image):
        thresh_val, thresh = cv2.threshold(
            image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )
        kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
        result = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
        return result, thresh_val

    img_denoised, thresh_val = denoise(img_torus)

    if debug:
        print(f"[DET] Otsu threshold = {thresh_val:.1f}")
        n_bialych = int((img_denoised > 127).sum())
        print(f"[DET] img_denoised: białych={n_bialych} "
              f"({100*n_bialych/img_denoised.size:.2f}%)")

    if SAVE_KLATKA_TORUS:
        debug_bw_path = path + f'debug_bw_i{i}_torus.png'
        cv2.imwrite(debug_bw_path, img_denoised)
        print(f"[DET] Zapisano: {debug_bw_path}")

    t0 = time.time()
    blobs_log = blob_log(img_denoised, min_sigma=BLOB_MIN_SIGMA,
                         max_sigma=BLOB_MAX_SIGMA, num_sigma=BLOB_NUM_SIGMA,
                         threshold=BLOB_THRESHOLD)
    blobs_dog = blob_dog(img_denoised, min_sigma=BLOB_MIN_SIGMA,
                         max_sigma=BLOB_MAX_SIGMA,
                         threshold=BLOB_THRESHOLD)
    if len(blobs_log) > 0:
        blobs_log[:, 2] = blobs_log[:, 2] * sqrt(2)
    if len(blobs_dog) > 0:
        blobs_dog[:, 2] = blobs_dog[:, 2] * sqrt(2)
    t_detekcja = time.time() - t0

    if debug:
        print(f"[DET] blobs_log: N={len(blobs_log)}")
        print(f"[DET] blobs_dog: N={len(blobs_dog)}")
        print(f"[DET] detekcja: {t_detekcja:.2f} s")

    def filtruj_promienie(blobs, r_min, r_max):
        if len(blobs) == 0:
            return blobs
        mask = (blobs[:, 2] >= r_min) & (blobs[:, 2] <= r_max)
        return blobs[mask]

    blobs_log_f = filtruj_promienie(blobs_log, r_min_detekcji, r_max_detekcji)
    blobs_dog_f = filtruj_promienie(blobs_dog, r_min_detekcji, r_max_detekcji)

    if len(blobs_log_f) > 0 and len(blobs_dog_f) > 0:
        blobs_sum = np.concatenate([blobs_log_f[:, :2], blobs_dog_f[:, :2]], axis=0)
    elif len(blobs_log_f) > 0:
        blobs_sum = blobs_log_f[:, :2]
    elif len(blobs_dog_f) > 0:
        blobs_sum = blobs_dog_f[:, :2]
    else:
        blobs_sum = np.zeros((0, 2))

    if len(blobs_sum) > 0:
        mask_oryginal = (
            (blobs_sum[:, 0] >= delta) & (blobs_sum[:, 0] < delta + H) &
            (blobs_sum[:, 1] >= delta) & (blobs_sum[:, 1] < delta + W)
        )
        blobs_sum = blobs_sum[mask_oryginal]

    if len(blobs_sum) > 0:
        blobs_sum = blobs_sum.copy()
        blobs_sum[:, 0] -= delta
        blobs_sum[:, 1] -= delta

    if len(blobs_sum) > 0:
        blobs_sum = clean_kdtree_torus(blobs_sum, 0, 0, y_w, x_w, r_min,
                                        label=f"clean_torus i={i}")

    if debug:
        print(f"[DET] Po odrzuceniu pasków i clean: N={len(blobs_sum)}")

    _blobs_cache[i] = blobs_sum
    return blobs_sum


# =====================================================================
# USUNIĘCIE DUPLIKATÓW TORUSOWYCH — bez uśredniania
# =====================================================================

def usun_duplikaty_torus(blobs_roi, y_d, x_d, r_min):
    """
    Usuwa duplikaty torusowe: jeśli dwa punkty są bliższe niż r_min
    w metryce torusowej, USUWA ten z pary, który ma WIĘKSZE współrzędne
    (y, x). Zostaje punkt z mniejszymi współrzędnymi (bliżej (0, 0)).

    Alternatywa dla symetryzuj_torus (które uśrednia — tworzy sztuczne punkty).

    Parametry:
        blobs_roi : (N, 2) — punkty w układzie lokalnym okna [0, y_d) × [0, x_d)
        y_d, x_d  : rozmiar okna (okres torusa)
        r_min     : próg scalania (px)

    Zwraca:
        (M, 2) — punkty po usunięciu duplikatów, M ≤ N.
    """
    import numpy as np
    from scipy.spatial import cKDTree

    blobs_roi = np.asarray(blobs_roi, dtype=np.float64)
    if len(blobs_roi) < 2:
        return blobs_roi

    N = len(blobs_roi)

    # 9 kopii punktów — z oryginalnym indeksem
    shifts = [
        (0, 0), (0, +x_d), (0, -x_d),
        (+y_d, 0), (-y_d, 0),
        (+y_d, +x_d), (+y_d, -x_d),
        (-y_d, +x_d), (-y_d, -x_d),
    ]
    kopie_lista, orig_idx_lista = [], []
    for k, (dy, dx) in enumerate(shifts):
        kopia = blobs_roi.copy()
        kopia[:, 0] += dy
        kopia[:, 1] += dx
        kopie_lista.append(kopia)
        orig_idx_lista.append(np.arange(N, dtype=np.int32))
    kopie = np.concatenate(kopie_lista)
    orig_idx_all = np.concatenate(orig_idx_lista)

    # cKDTree na 9N punktach
    tree_k = cKDTree(kopie)
    pairs = tree_k.query_pairs(r=r_min, output_type='ndarray')

    if len(pairs) == 0:
        print(f"[usun_duplikaty] Brak par < {r_min}. N={N}")
        return blobs_roi

    # Dla każdej pary — oznacz do usunięcia ten o WIĘKSZYCH współrzędnych
    do_usuniecia = np.zeros(N, dtype=bool)

    for a, b in pairs:
        orig_a = orig_idx_all[a]
        orig_b = orig_idx_all[b]

        if orig_a == orig_b:
            continue  # ten sam oryginał (kopia samego siebie)

        p_a = blobs_roi[orig_a]
        p_b = blobs_roi[orig_b]

        # Porównanie leksykograficzne (y, x)
        if (p_a[0] > p_b[0]) or (p_a[0] == p_b[0] and p_a[1] > p_b[1]):
            do_usuniecia[orig_a] = True
        else:
            do_usuniecia[orig_b] = True

    wynik = blobs_roi[~do_usuniecia]
    n_usunietych = int(do_usuniecia.sum())
    print(f"[usun_duplikaty] {N} → {len(wynik)} (usunięto {n_usunietych})")
    return wynik


# =====================================================================
# MACIERZ p_weierstrass BEZ dodatkowego π
# =====================================================================

def p_weierstrass_matrix(z, g2=189.0727, g3=0, block_cpu=10_000):
    """Macierz p_weierstrass(z) BEZ dodatkowej π (diagonala = 0).
    ZMIANA: liczone wektorowo w pfast.py (ta sama formuła theta, stałe sieci liczone raz)
    zamiast element po elemencie w mpmath. block_cpu nieużywany (zgodność wywołań)."""
    from pfast import p_weierstrass_matrix_fast
    return p_weierstrass_matrix_fast(z, g2=g2, g3=g3)


# =====================================================================
# PREPARE
# =====================================================================

def prepare(x, y, i, save_png_for_i=1, png_prefix='real',
            dist_scalanie=10, debug_nr=2, debug_mr=2):
    """Dla okna (nr=y, mr=x) klatki i: bloby + usunięcie duplikatów + e2/e22 + PNG/TXT/CSV."""
    import numpy as np
    from pfast import cp   # ZMIANA: CuPy na GPU, NumPy bez GPU
    from mpmath import pi

    y_0 = y_0_min + y * krok_y
    x_0 = x_0_min + x * krok_x

    blobs_sum = wykryj_bloby_na_torusie(i, debug=(i == save_png_for_i and
                                                   x == debug_mr and y == debug_nr))

    if len(blobs_sum) == 0:
        print(f"Brak detekcji dla i={i}, nr={y}, mr={x} — pomijam.")
        if i == save_png_for_i:
            _, img_bez_paskow = przygotuj_torus(i)
            roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
            _zapisz_png(roi, np.zeros((0, 2)), i, y, x, png_prefix)
        return None

    mask_okno = (
        (blobs_sum[:, 0] >= y_0) & (blobs_sum[:, 0] < y_0 + y_d) &
        (blobs_sum[:, 1] >= x_0) & (blobs_sum[:, 1] < x_0 + x_d)
    )
    blobs_okno = blobs_sum[mask_okno].copy()

    if len(blobs_okno) == 0:
        print(f"Brak blobów w oknie i={i}, nr={y}, mr={x} — pomijam.")
        if i == save_png_for_i:
            _, img_bez_paskow = przygotuj_torus(i)
            roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
            _zapisz_png(roi, np.zeros((0, 2)), i, y, x, png_prefix)
        return None

    blobs_okno[:, 0] -= y_0
    blobs_okno[:, 1] -= x_0

    # --- 4. Usunięcie duplikatów torusowych (zamiast uśredniania) ---
    blobs_roi = usun_duplikaty_torus(blobs_okno, y_d, x_d, r_min)

    # Opcjonalna kwantyzacja (bez przesuwania układu — y_0 = x_0 = 0)
    if len(blobs_roi) > 1:
        blobs_roi = kwantyzuj_do_siatki(blobs_roi, 0, 0, y_d, x_d, krok=dist_scalanie)

    if len(blobs_roi) == 0:
        print(f"Po usunięciu duplikatów brak punktów i={i}, nr={y}, mr={x} — pomijam.")
        if i == save_png_for_i:
            _, img_bez_paskow = przygotuj_torus(i)
            roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
            _zapisz_png(roi, np.zeros((0, 2)), i, y, x, png_prefix)
        return None

    print(f"[DIAG] i={i} nr={y} mr={x}: sum={len(blobs_sum)}, "
          f"okno={len(blobs_okno)}, roi={len(blobs_roi)}")

    if i == save_png_for_i:
        _, img_bez_paskow = przygotuj_torus(i)
        roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
        _zapisz_png(roi, blobs_roi, i, y, x, png_prefix)

    # --- Normalizacja do [0, 1] + limit ---
    blobs_sum2 = blobs_roi.copy()
    blobs_sum2[:, 0] = blobs_roi[:, 0] / y_d
    blobs_sum2[:, 1] = blobs_roi[:, 1] / x_d

    N_roi = len(blobs_sum2)
    if N_roi > MAX_PUNKTOW_E2:
        print(f"[LIMIT] blobs_sum2 ma {N_roi} punktów > {MAX_PUNKTOW_E2} — losuję podzbiór.")
        idx = np.random.choice(N_roi, MAX_PUNKTOW_E2, replace=False)
        blobs_sum2 = blobs_sum2[idx]

    # --- e2 + e22 z WSPÓŁDZIELONĄ macierzą P_matrix ---
    from mpmath import mp
    mp.dps = 15

    array = cp.asarray(blobs_sum2)
    N = len(array)

    M_e2 = 1 / (N ** 2)
    p_local = 2
    M_e22 = ((-1) ** p_local) / (N ** (p_local + 1))

    # Mathematica: a1 = x + I*y  (array[:,0] = y_norm, array[:,1] = x_norm)
    a = cp.asarray(array[:, 1] + 1j * array[:, 0], dtype=cp.complex128)
    t0_e2 = time.time()

    diffs = a[:, None] - a[None, :]
    cp.fill_diagonal(diffs, cp.complex128(0))

    P_matrix = p_weierstrass_matrix(diffs, g2=189.0727, g3=0, block_cpu=10_000)

    f2 = cp.sum(P_matrix)
    e2 = cp.complex128(pi()) + M_e2 * f2

    E2_matrix = P_matrix + cp.complex128(pi())
    fpp = cp.sum(E2_matrix, axis=1)
    e22 = M_e22 * cp.sum(cp.abs(fpp) ** 2)

    t_e2 = time.time() - t0_e2
    e22_val = float(cp.asnumpy(e22).real) if hasattr(e22, 'real') else float(e22)

    # --- LEPKOŚĆ: e33, e4, A3, mu_e/mu dla tego okna (komórka 3d) ---   # <-- DODANE
    try:                                                                # <-- DODANE
        lep = lepkosc_okna(a, cp)                                        # <-- DODANE
        if lep is not None:                                              # <-- DODANE
            WYNIKI_LEPKOSC.append(dict(i=i, nr=y, mr=x, y0=y_0, x0=x_0, **lep))   # <-- DODANE
    except Exception as e:                                               # <-- DODANE
        print(f"[LEPK] okno i={i} nr={y} mr={x}: {e}")                   # <-- DODANE

    wynik = f"N={N}, okno: {y_0},{x_0}, e2={e2}, e22={e22_val}\n"
    print(wynik)
    gpu_mem_info(f"i={i} nr={y} mr={x} po e22")
    print(f"[TIMING] i={i} nr={y} mr={x} | e2+e22={t_e2:.2f}s | N={N}")
    return wynik


# =====================================================================
# PRZETWORZENIE JEDNEJ KLATKI
# =====================================================================

def przetworz_klatke(i):
    """Pełny potok dla klatki i."""
    t_start = time.time()
    print(f"\n{'='*70}")
    print(f"OBRAZEK i={i}  ({i - i_min + 1}/{i_max - i_min + 1})")
    print(f"{'='*70}")

    data_txt = PATH_TXT + TXT_PREFIX + str(i) + '.txt'
    data_csv = PATH_CSV + TXT_PREFIX + str(i) + '.csv'

    print(f"[KROK 1+2] Przygotowanie torusa i detekcja...")
    przygotuj_torus(i)
    wykryj_bloby_na_torusie(i, debug=True)

    with open(data_txt, "w", encoding='utf-8') as f:
        f.write(f"# Analiza obrazka i={i} (klatka_{i:03d}-torus.png)\n")
        f.write(f"# Oryginał: {y_w_orig}×{x_w_orig}, obcięcie [{Y_CUT_TOP}:"
                f"{y_w_orig - Y_CUT_BOT}], po obcięciu {y_w}×{x_w}\n")
        f.write(f"# Torus: {y_w_torus}×{x_w_torus} (delta={delta})\n")
        f.write(f"# ROI: {y_d}×{x_d}\n")
        f.write(f"# Okna: nr ∈ [{nr_min},{nr_max}] ({LICZBA_OKIEN_Y}), "
                f"mr ∈ [{mr_min},{mr_max}] ({LICZBA_OKIEN_X})\n")
        f.write(f"# y_0 ∈ [{y_0_min},{y_0_max}], krok_y={krok_y}\n")
        f.write(f"# x_0 ∈ [{x_0_min},{x_0_max}], krok_x={krok_x}\n")
        f.write(f"# r_min={r_min}, delta={delta}\n")
        f.write(f"# Kwantyzacja: DIST_SCALANIE={DIST_SCALANIE} px\n")
        f.write(f"# Detektory: sigma ∈ [{BLOB_MIN_SIGMA}, {BLOB_MAX_SIGMA}], "
                f"num_sigma={BLOB_NUM_SIGMA}, threshold={BLOB_THRESHOLD}\n")
        f.write(f"# Format: N, okno_y0,x0, e2, e22\n\n")

    WYNIKI_LEPKOSC.clear()   # <-- DODANE
    n_okien_ok = 0
    n_okien_total = 0
    for nr in range(nr_max, nr_min - 1, -1):
        for mr in range(mr_max, mr_min - 1, -1):
            n_okien_total += 1
            print(f"---><--- i={i}, nr={nr}, mr={mr} ---><---")
            try:
                wynik = prepare(mr, nr, i,
                                save_png_for_i=SAVE_PNG_FOR_I,
                                png_prefix=PNG_PREFIX,
                                dist_scalanie=DIST_SCALANIE,
                                debug_nr=DEBUG_NR,
                                debug_mr=DEBUG_MR)
                if wynik is not None:
                    with open(data_txt, "a", encoding='utf-8') as f:
                        f.write(wynik)
                    n_okien_ok += 1
            except Exception as e:
                print(f"BŁĄD dla i={i}, nr={nr}, mr={mr}: {e}")
                with open(data_txt, "a", encoding='utf-8') as f:
                    f.write(f"# BŁĄD i={i}, nr={nr}, mr={mr}: {e}\n")
                try:
                    import cupy as cp
                    cp.get_default_memory_pool().free_all_blocks()
                except Exception:
                    pass

    t_txt = time.time()
    print(f"\n[KROK 3] TXT zapisany: {data_txt}  "
          f"({n_okien_ok}/{n_okien_total} okien, {t_txt - t_start:.1f} s)")

    n_wierszy = konwertuj_txt_na_csv(data_txt, data_csv)
    if n_wierszy is None:
        print(f"[KROK 4] Nie udało się utworzyć CSV dla i={i}")
        return {'i': i, 'txt': False, 'csv': False, 'png_3d': 0, 'png_2d': 0,
                'stat': False, 'czas': time.time() - t_start}

    t_csv = time.time()
    print(f"[KROK 4] CSV zapisany: {data_csv}  ({n_wierszy} wierszy, "
          f"{t_csv - t_txt:.1f} s)")

    n_3d, n_2d, bledy = zapisz_wizualizacje(data_csv, i)
    n2c, bledy_c = zapisz_wizualizacje_koncentracji(data_csv, i)   # <-- DODANE
    print(f"[KROK 5b] Koncentracja: {n2c}×2D, błędów: {len(bledy_c)}")   # <-- DODANE
    try:                                                                  # <-- DODANE
        pd.DataFrame(WYNIKI_LEPKOSC).to_csv(PATH_CSV + f'real{i}_lepkosc_okna.csv', index=False)   # <-- DODANE
        zapisz_lepkosc_efektywna(data_csv, i)                              # <-- DODANE
    except Exception as e:                                                 # <-- DODANE
        print(f"[LEPK] Błąd dla i={i}: {e}")                             # <-- DODANE
    try:                                                                  # <-- v17: TXT z oknami klatki
        zapisz_okna_txt(i)                                                 # (funkcja z komórki 3e)
    except NameError:
        print("[A456] brak komórki 3e — pomijam zapis real{i}_lepkosc_okna.txt")
    except Exception as e:
        print(f"[A456] Błąd zapisu TXT dla i={i}: {e}")
    t_png = time.time()
    print(f"[KROK 5] PNG zapisane: {n_3d}×3D + {n_2d}×2D, "
          f"{t_png - t_csv:.1f} s")
    if bledy:
        print(f"[KROK 5] Błędy: {len(bledy)}")
        for j, typ, err in bledy[:3]:
            print(f"    i={j}, {typ}: {err}")

    ok_stat = zapisz_statystyki_dla_klatki(i)
    t_stat = time.time()

    czas_total = time.time() - t_start
    print(f">>> Klatka i={i} zakończona w {czas_total:.1f} s "
          f"(TXT {t_txt - t_start:.1f}s + CSV {t_csv - t_txt:.1f}s + "
          f"PNG {t_png - t_csv:.1f}s + STAT {t_stat - t_png:.1f}s)")

    return {'i': i, 'txt': True, 'csv': True,
            'png_3d': n_3d, 'png_2d': n_2d,
            'stat': ok_stat,
            'czas': czas_total}

In [ ]:
# Komórka 3c: koncentracja N — interpolacja i ContourPlot 2D (3D tylko dla klatek z anomaliami: komórka 7)

INTERP_METHOD = 'linear' # interpolacja: 'linear' lub 'cubic'
CONC_LABEL    = 'N (liczba bakterii w oknie)'
CONC_VLIM     = None     # np. (0, 300) = wspólna skala kolorów dla wszystkich klatek


def _pelna_siatka_N(df):
    """Pełna siatka okien (y_0, x_0); okna bez bakterii (brak wiersza) -> N = 0."""
    ys = np.array([y_0_min + k * krok_y for k in range(LICZBA_OKIEN_Y)])
    xs = np.array([x_0_min + k * krok_x for k in range(LICZBA_OKIEN_X)])
    piv = (df.pivot_table(index='y', columns='x', values='N', aggfunc='first')
             .reindex(index=ys, columns=xs))
    n_puste = int(piv.isna().sum().sum())
    piv = piv.fillna(0)
    Xg, Yg = np.meshgrid(xs, ys)
    return (Xg.ravel().astype(float), Yg.ravel().astype(float),
            piv.to_numpy(dtype=float).ravel(), n_puste)


def _interpoluj_N(x, y, z):
    """Siatka GRID_N x GRID_N i interpolacja N (poza otoczką punktów: najbliższy sąsiad)."""
    Xg, Yg = np.meshgrid(np.linspace(x.min(), x.max(), GRID_N),
                         np.linspace(y.min(), y.max(), GRID_N))
    Z = griddata((x, y), z, (Xg, Yg), method=INTERP_METHOD)
    brak = np.isnan(Z)
    if brak.any():
        Z[brak] = griddata((x, y), z, (Xg, Yg), method='nearest')[brak]
    return Xg, Yg, Z


def _rysuj_2d_z_siatki(x, y, Xg, Yg, Zg, z_label, tytul):
    if CONC_VLIM:
        lev_f = np.linspace(CONC_VLIM[0], CONC_VLIM[1], 31)
        lev_c = np.linspace(CONC_VLIM[0], CONC_VLIM[1], 16)
        ext = 'both'
    else:
        lev_f, lev_c, ext = 30, 15, 'neither'
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    cf = ax.contourf(Xg, Yg, Zg, levels=lev_f, cmap='viridis', extend=ext)
    ct = ax.contour(Xg, Yg, Zg, levels=lev_c, colors='black', linewidths=0.5, alpha=0.6)
    ax.clabel(ct, inline=True, fontsize=7, fmt='%.3g')
    ax.scatter(x, y, c='red', s=8, marker='o', linewidths=0, zorder=3)
    fig.colorbar(cf, ax=ax, shrink=0.8).set_label(z_label)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(f"ContourPlot: {tytul}")
    ax.set_aspect('equal')
    plt.tight_layout()
    return fig


def zapisz_wizualizacje_koncentracji(csv_file, i):
    """1 PNG na klatkę: ContourPlot 2D interpolacji N (real{i}_2D_N_interp.png). Zwraca (n_2d, błędy)."""
    df = _wczytaj_csv(csv_file)
    if df is None:
        return 0, []

    x, y, z, n_puste = _pelna_siatka_N(df)
    Xg, Yg, Z_int = _interpoluj_N(x, y, z)
    print(f"[KONC] i={i}: okien={len(z)} (puste->0: {n_puste}), N: min={z.min():.0f}, max={z.max():.0f}")

    try:
        fig = _rysuj_2d_z_siatki(x, y, Xg, Yg, Z_int, CONC_LABEL,
                                 f"{CONC_LABEL}, interpolacja ({INTERP_METHOD})")
        fig.savefig(PATH_PNG + f'real{i}_2D_N_interp.png', dpi=100)
        plt.close(fig)
        return 1, []
    except Exception as e:
        plt.close('all')
        return 0, [(i, "2D N interp", str(e))]

In [ ]:
# Komórka 3d (v4): lepkość efektywna z sum strukturalnych e33, e4 — dla KAŻDEGO okna
# Wzory jak w F.tex: A3 = (2/pi^2)(2 e33 + 3 e4),  Re A3 i Im A3 (e33 rzeczywiste => Im A3 = (6/pi^2) Im e4),  mu_e/mu = (1 + f + A3 f^3)/(1 - f - A3 f^3)
# Wklej PO komórce 3c i PRZED pętlą główną (komórka 4).
# Potrzebuje: math, np, pd, plt (komórka 1) oraz funkcji z 3a/3c:
#   _wczytaj_csv, _pelna_siatka_N, _interpoluj_N, _rysuj_2d_z_siatki

# =====================================================================
# PARAMETRY
# =====================================================================
R_BAKT        = 3.5    # <-- r_px: promień bakterii [px] używany w f (zmieniaj ręcznie).
                       # Wzory F.tex (rozwinięcie dla nienakładających się kół) mają sens, gdy 2*r_px <= minimalny
                       # odstęp środków (czyszczenie: r_min = 8 / DIST_SCALANIE = 10 px), czyli r_px <= ok. 4.
N_WYSOKIE     = 2      # ile miejsc o wysokim N
N_NISKIE      = 2      # ile miejsc o niskim N
DIST_PUNKTOW  = 110    # min. odległość między wybranymi miejscami [px]
MIN_N_OKNA    = 3      # okna z mniejszą liczbą punktów pomijamy

# Wzór (1) z pracy ECMS 2024 (sztywne koła, losowo) — porównawczo
MU_COEF = [1.0, 2.0, 2.0, 3.8114, 8.2731, 5.3345, 20.5096]

# Wyniki z okien bieżącej klatki (wypełnia prepare(), zapisuje przetworz_klatke())
WYNIKI_LEPKOSC = []

# Stałe sieci kwadratowej o okresach 1 i i  (Part I (A5), Part II (30), (A7))
_G14  = math.gamma(0.25)
_S4   = _G14 ** 8 / (960 * math.pi ** 2)      # 3.151211
_G2   = 60.0 * _S4                             # 189.0727
_S31  = _G14 ** 8 / (384 * math.pi ** 3)      # S3^(1) = 2.50765
_QNOM = math.exp(-math.pi)                     # nome dla tau = i


# =====================================================================
# FUNKCJE ELIPTYCZNE (szereg theta; xp = numpy albo cupy)
# =====================================================================
def wp_zeta_dp(z, xp):
    """zeta, wp, wp' sieci kwadratowej (okresy 1, i). |Re z|,|Im z| <= 0.5 (najbliższy obraz)."""
    u = math.pi * z
    t0 = t1 = t2 = t3 = 0 * u
    for n in range(8):
        a = 2 * n + 1
        c = 2.0 * (-1) ** n * _QNOM ** ((n + 0.5) ** 2)
        s, co = xp.sin(a * u), xp.cos(a * u)
        t0 = t0 + c * s
        t1 = t1 + c * a * co
        t2 = t2 - c * a ** 2 * s
        t3 = t3 - c * a ** 3 * co
    L1 = t1 / t0
    L2 = t2 / t0 - L1 ** 2
    L3 = t3 / t0 - t2 * t1 / t0 ** 2 - 2 * L1 * L2
    zeta = math.pi * z + math.pi * L1
    wp = -math.pi - math.pi ** 2 * L2
    dwp = -math.pi ** 3 * L3
    return zeta, wp, dwp


# =====================================================================
# SUMY STRUKTURALNE e33^{(1,1)(1,0)} i e4^{(0)(1)}  (Part II: (39), (40), (A7))
# =====================================================================
def sumy_e33_e4(a, xp):
    """a: zespolone położenia środków w kwadracie jednostkowym (1-D). Zwraca (e33, Re e4, Im e4)."""
    N = int(a.shape[0])
    d = a[:, None] - a[None, :]
    d = d - xp.round(d.real) - 1j * xp.round(d.imag)          # najbliższy obraz torusowy
    eye = xp.eye(N, dtype=bool)
    d = xp.where(eye, 0.3 + 0.3j, d)                           # unikamy z=0 w szeregach
    zeta, wp, dwp = wp_zeta_dp(d, xp)

    E3 = -dwp / 2.0
    E4 = wp ** 2 - _G2 / 12.0                                  # = wp''/6
    E31 = (xp.conj(d) - zeta / math.pi) * E3 + E4 / math.pi - 5.0 * _S4 / math.pi + _S31
    E31 = xp.where(eye, 0.0, E31)                              # E3^(1)(0) := 0
    E4 = xp.where(eye, _S4, E4)                                # E4(0)    := S4

    e33 = float(xp.sum(xp.abs(xp.sum(E31, axis=1)) ** 2)) / N ** 3   # F.tex (11): bez znaku minus
    e4 = complex(xp.sum(E4)).conjugate() / N ** 3              # F.tex (12): e4^(0)(1) = (1/N^3) sum conj(E4)
    return e33, e4.real, e4.imag


def stezenie_f(N, r_px=None):
    r_px = R_BAKT if r_px is None else r_px
    return N * math.pi * (r_px / x_d) ** 2                     # kwadrat jednostkowy: r = r_px / 220


def mu_eff_szereg(f):
    return sum(c * f ** k for k, c in enumerate(MU_COEF))


def lepkosc_okna(a, xp):
    """Jedno okno: e33, Re e4, Re A3, f, mu_e/mu wg F.tex.
    (13): A3 = (2/pi^2)(2 e33 + 3 e4');  (14): mu_e/mu = (1+s)/(1-s), s = f + A3 f^3;
    mu_e_wz15: wzór (15) z F.tex, s = f + (e33 + e4') f^3 (tylko porównawczo)."""
    N = int(a.shape[0])
    if N < MIN_N_OKNA:
        return None
    e33, e4r, e4i = sumy_e33_e4(a, xp)
    A3 = 2.0 / math.pi ** 2 * (2.0 * e33 + 3.0 * e4r)          # Re A3 (e33 jest rzeczywiste)
    A3_im = 6.0 / math.pi ** 2 * e4i                           # Im A3 = (2/pi^2) * 3 * Im e4
    f = stezenie_f(N)
    s = f + A3 * f ** 3
    mu = (1.0 + s) / (1.0 - s) if s < 1.0 else float('nan')
    s15 = f + (e33 + e4r) * f ** 3
    mu15 = (1.0 + s15) / (1.0 - s15) if s15 < 1.0 else float('nan')
    return dict(N=N, f=f, e33=e33, Re_e4=e4r, Im_e4=e4i, Re_A3=A3, Im_A3=A3_im,
                mu_e_pade=mu, mu_e_wz15=mu15, mu_e_szereg=mu_eff_szereg(f))


# =====================================================================
# WYBÓR MIEJSC I RYSOWANIE
# =====================================================================
def _wybierz_ekstrema(Xg, Yg, Zg, n_wys, n_nis, dmin):
    """Zachłannie n_wys najwyższych i n_nis najniższych punktów pola, każdy >= dmin od poprzednich."""
    xs, ys, zs = Xg.ravel(), Yg.ravel(), Zg.ravel()
    kolejnosc = np.argsort(zs)

    def zachlannie(order, n):
        wybrane = []
        for k in order:
            if all(np.hypot(xs[k] - qx, ys[k] - qy) >= dmin for qx, qy in wybrane):
                wybrane.append((float(xs[k]), float(ys[k])))
            if len(wybrane) == n:
                break
        return wybrane

    return zachlannie(kolejnosc[::-1], n_wys), zachlannie(kolejnosc, n_nis)


def _wiersz_okna(tab, px, py):
    """Wiersz tabeli okien najbliższy (x0, y0) = (px, py)."""
    d = (tab['x0'] - px) ** 2 + (tab['y0'] - py) ** 2
    return tab.loc[d.idxmin()]


from matplotlib.patches import Rectangle


def zapisz_lepkosc_efektywna(csv_file, i):
    """Dla klatki i: dwa okna pływające o największym i dwa o najmniejszym N (pole interpolowane).
    Na mapie rysujemy kwadraty tych okien (x_d x y_d); w etykiecie: N, f, Re A3, Im A3, mu_e/mu tego okna -> PNG + CSV."""
    df = _wczytaj_csv(csv_file)
    tab_file = PATH_CSV + f'real{i}_lepkosc_okna.csv'
    if df is None or not os.path.isfile(tab_file):
        print(f"[LEPK] brak danych dla i={i}: {csv_file} / {tab_file}")
        return None
    try:
        tab = pd.read_csv(tab_file)
    except Exception as e:
        print(f"[LEPK] i={i}: pusty/nieczytelny {tab_file} ({e}). Sprawdź w logu linie '[LEPK] okno ...' "
              f"(uruchom ponownie komórki 3b i 3d, potem 4).")
        return None
    if tab.empty or 'Im_A3' not in tab.columns:
        print(f"[LEPK] i={i}: {tab_file} nie zawiera wyników z nowego kodu (brak kolumny Im_A3). "
              f"Uruchom ponownie komórki 3b, 3d i 4 w tej kolejności.")
        return None
    n_nan = int(tab['mu_e_pade'].isna().sum())
    print(f"[LEPK] i={i}: okien z wynikiem: {len(tab)}, w tym bez mu_e/mu (s >= 1): {n_nan}, "
          f"z f >= 1: {int((tab['f'] >= 1).sum())}")
    if 2 * R_BAKT > min(r_min, DIST_SCALANIE):
        print(f"[LEPK] UWAGA: 2*r_px = {2 * R_BAKT} px > minimalny odstęp środków (~{min(r_min, DIST_SCALANIE)} px): "
              f"koła bakterii się nakładają, wzór F.tex (14) jest poza zakresem (f >= 1 dla N >= "
              f"{math.ceil(1 / (math.pi * (R_BAKT / x_d) ** 2))}); dla r_px <= {min(r_min, DIST_SCALANIE) / 2:g} "
              f"mu_e/mu liczy się poprawnie.")

    x, y, z, _ = _pelna_siatka_N(df)
    # mapa we współrzędnych ŚRODKÓW okien (x0 + x_d/2, y0 + y_d/2): kwadrat okna jest wtedy wycentrowany na swoim punkcie
    sx, sy = x_d / 2, y_d / 2
    x = x + sx
    y = y + sy
    Xg, Yg, Z_int = _interpoluj_N(x, y, z)

    wys, nis = _wybierz_ekstrema(Xg, Yg, Z_int, N_WYSOKIE, N_NISKIE, DIST_PUNKTOW)
    if len(wys) < N_WYSOKIE or len(nis) < N_NISKIE:
        print(f"[LEPK] UWAGA: znaleziono H={len(wys)}/{N_WYSOKIE}, L={len(nis)}/{N_NISKIE} miejsc "
              f"(zmniejsz DIST_PUNKTOW).")
    miejsca = ([('wysokie', f'H{k + 1}', px, py) for k, (px, py) in enumerate(wys)] +
               [('niskie',  f'L{k + 1}', px, py) for k, (px, py) in enumerate(nis)])

    fig = _rysuj_2d_z_siatki(x, y, Xg, Yg, Z_int, CONC_LABEL,
                             f"{CONC_LABEL}, interpolacja ({INTERP_METHOD})\n"
                             f"H = okna o wysokim N, L = okna o niskim N; kwadrat = okno pływające "
                             f"{x_d}x{y_d} px; r = {R_BAKT} px")
    ax = fig.axes[0]
    ax.set_aspect('equal', adjustable='box')   # okna mają wyglądać jak kwadraty
    ax.set_xlim(x.min() - sx, x.max() + sx)    # kwadraty okien wystają poza pole o pół okna
    ax.set_ylim(y.min() - sy, y.max() + sy)
    ax.set_xlabel('x [px] (środek okna)')
    ax.set_ylabel('y [px] (środek okna)')
    wiersze = []
    for typ, nazwa, px, py in miejsca:
        w = _wiersz_okna(tab, px - sx, py - sy)   # px, py: współrzędne mapy -> początek okna
        wiersze.append(dict(i=i, typ=typ, miejsce=nazwa,
                            x_okna=w['x0'], y_okna=w['y0'],
                            x_srodek=w['x0'] + sx, y_srodek=w['y0'] + sy,
                            N=w['N'], f=w['f'], e33=w['e33'], Re_e4=w['Re_e4'], Im_e4=w['Im_e4'],
                            Re_A3=w['Re_A3'], Im_A3=w['Im_A3'], mu_e_przez_mu=w['mu_e_pade'],
                            mu_e_wz15=w['mu_e_wz15'], mu_e_szereg=w['mu_e_szereg'],
                            Re_A4=w.get('Re_A4', np.nan), Re_A5=w.get('Re_A5', np.nan),
                            Re_A6=w.get('Re_A6', np.nan), mu_e_A4=w.get('mu_e_A4', np.nan),
                            mu_e_A5=w.get('mu_e_A5', np.nan), mu_e_A6=w.get('mu_e_A6', np.nan)))   # v17: kolumny z komórki 3e
        # kwadrat okna pływającego: [x0, x0 + x_d] x [y0, y0 + y_d]
        ax.add_patch(Rectangle((w['x0'], w['y0']), x_d, y_d, fill=False,
                               edgecolor='white', linewidth=4.0, zorder=5))
        ax.add_patch(Rectangle((w['x0'], w['y0']), x_d, y_d, fill=False,
                               edgecolor='black', linewidth=1.8, zorder=6))
        mu_txt = f"{w['mu_e_pade']:.3f}" if np.isfinite(w['mu_e_pade']) else "brak (s≥1)"
        et = (f"{nazwa}: N={w['N']:.0f}, f={w['f']:.3f}\n"
              f"Re A3={w['Re_A3']:.2f}, Im A3={w['Im_A3']:.2f}\n"
              f"μe/μ={mu_txt}")
        ax.annotate(et, (w['x0'] + sx, w['y0'] + sy), ha='center', va='center',
                    fontsize=8, zorder=7,
                    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='black', alpha=0.85))
    fname = PATH_PNG + f'real{i}_2D_N_interp_lepkosc.png'
    fig.savefig(fname, dpi=100)
    plt.close(fig)
    print(f"[LEPK] Zapisano: {fname}")

    out = pd.DataFrame(wiersze)
    csv_out = PATH_CSV + f'real{i}_lepkosc.csv'
    out.to_csv(csv_out, index=False, float_format='%.6g')
    print(f"[LEPK] Zapisano: {csv_out}")
    print(out[['miejsce', 'x_okna', 'y_okna', 'N', 'f', 'Re_A3', 'Im_A3', 'mu_e_przez_mu']]
          .to_string(index=False))
    return out

In [ ]:
# Komórka 3e (v17): współczynniki A4, A5, A6 z sum strukturalnych — dla KAŻDEGO okna
# Wzory (41), (42), (43) z: Czapla, Drygaś, Gluzman, Ligocki, Mityushev, "Computational Strategy for Analyzing
# Effective Properties of Random Composites—Part II: Elasticity", Materials 2025, 18, 5041 (plik elasticity2.tex).
# Funkcje _A4_wzor, _A5_wzor, _A6_wzor poniżej zostały WYGENEROWANE AUTOMATYCZNIE z elasticity2.tex (bez ręcznego
# przepisywania); suma e(n, j, l) = e_{n1..np}^{(j1..jp)(l1..lp)} to Def. sumy strukturalnej (wzór (33) w pracy).
# Wklej PO komórce 3d (v4) i PRZED pętlą główną (komórka 4).
# Potrzebuje: math, np (komórka 1) oraz wp_zeta_dp, lepkosc_okna, _S4, _G2, _S31, stezenie_f (komórka 3d).
#
# Co robi:
#  - sumy strukturalne dowolnego rzędu liczy jako iloczyn macierz-wektor (algorytm łańcuchowy, koszt O(p N^2)):
#        e = N^{-(alpha/2+1)} * 1^T M_1 M_2 ... M_p 1,   (M_s)_{kl} = C^{l_s} E_{n_s}^{(j_s)}(a_k - a_l),
#  - funkcje E_n (n = 2..6) i E_n^{(1)} (n = 3..6) wyraża przez zeta, p, p' (wzory Eisensteina-Natanzona-Filsztyńskiego),
#  - dodaje do wyniku każdego okna: Re/Im A4, A5, A6 oraz mu_e/mu z obcięciem szeregu na f^4, f^5, f^6.
# Wyniki trafiają do tych samych plików co dotychczas (real{i}_lepkosc_okna.csv) + nowy plik tekstowy
# real{i}_lepkosc_okna.txt (kolumny rozdzielone spacjami) — szeregi czasowe zbiera komórka 5b.

_PI = math.pi                      # (nie importujemy nazwy 'pi', żeby nie zasłonić ewentualnego pi z mpmath)

# ---------------------------------------------------------------------------------------------------------------
# PARAMETRY FIZYCZNE I KONWENCJE
# ---------------------------------------------------------------------------------------------------------------
# Parametry kontrastu rho1, rho2, rho3 (Part II, (25)): dla sztywnych inkluzji w nieściśliwej cieczy lepkiej
# (bakterie = sztywne koła, kappa = kappa1 = 1, m = eta1/eta -> inf) wszystkie trzy są równe 1.
RHO1 = RHO2 = RHO3 = 1.0
KAPPA = 1.0                       # mu_e/mu = (1 + s)/(1 - KAPPA s),  s = Re sum_k A^(k) f^k
S2_E2 = 0.0                       # S2 w E2 = wp + S2: Part II (A9) "S2 = 0 must be assumed" (liczy się tylko gdy RHO1 != RHO3)
DIAG_E31 = 0.0                    # E3^(1)(0) := 0 (jak w v16 i Part II (27)); definicja sumy ma S3^(1) = _S31 (do eksperymentów)
LICZ_A456 = True                  # False: komórka nie zmienia wyników v16


# ---------------------------------------------------------------------------------------------------------------
# FUNKCJE E_n i E_n^(1)
# ---------------------------------------------------------------------------------------------------------------
# E_n = (-1)^n/(n-1)! d^{n-2} wp/dz^{n-2},  g3 = 0 (sieć kwadratowa):
#   E3 = -wp'/2, E4 = wp^2 - g2/12, E5 = -wp wp'/2, E6 = wp^3 - 3 g2 wp/20, E7 = -wp'(wp^2/2 - g2/40).
# E_n^(1)(z) = sum_P conj(z-P)/(z-P)^n  (Eisenstein-Natanzon-Filshtinsky). Znany wzór (A7) na E3^(1) różniczkujemy po z
# (conj z jak stała): d/dz E_n^(1) = -n E_{n+1}^(1). Daje to (c = 1/pi, zb = conj z):
#   E4^(1) = zb E4 + (4c/3) E5 - (c/3) wp E3 - c zeta E4
#   E5^(1) = zb E5 + (5c/3) E6 - (c/6) E3^2 - (c/2) wp E4 - c zeta E5
#   E6^(1) = zb E6 + 2c E7 - (2c/5) E3 E4 - (3c/5) wp E5 - c zeta E6
# (sprawdzone numerycznie z bezpośrednimi sumami sieciowymi: patrz _selftest_A456()).
def macierze_E(a, xp):
    """Słownik {(n, j): macierz E_n^(j)(a_k - a_l)} dla n = 2..6; diagonala = S_n^(j)."""
    N = int(a.shape[0])
    d = a[:, None] - a[None, :]
    d = d - xp.round(d.real) - 1j * xp.round(d.imag)          # najbliższy obraz torusowy
    eye = xp.eye(N, dtype=bool)
    d = xp.where(eye, 0.3 + 0.3j, d)                           # unikamy z = 0 w szeregach
    zeta, P, Q = wp_zeta_dp(d, xp)                             # zeta, wp, wp'
    zb = xp.conj(d)
    c = 1.0 / _PI
    E3 = -Q / 2.0
    E4 = P ** 2 - _G2 / 12.0
    E5 = -P * Q / 2.0
    E6 = P ** 3 - 3.0 * _G2 * P / 20.0
    E7 = -Q * (P ** 2 / 2.0 - _G2 / 40.0)
    E = {(2, 0): P + S2_E2, (3, 0): E3, (4, 0): E4, (5, 0): E5, (6, 0): E6,
         (3, 1): (zb - c * zeta) * E3 + c * E4 - 5.0 * c * _S4 + _S31,
         (4, 1): zb * E4 + 4 * c / 3 * E5 - c / 3 * P * E3 - c * zeta * E4,
         (5, 1): zb * E5 + 5 * c / 3 * E6 - c / 6 * E3 ** 2 - c / 2 * P * E4 - c * zeta * E5,
         (6, 1): zb * E6 + 2 * c * E7 - 2 * c / 5 * E3 * E4 - 3 * c / 5 * P * E5 - c * zeta * E6}
    # wartości w zerze E_n^(j)(0) := S_n^(j); dla sieci kwadratowej S_n^(j) = 0 poza S4 i S3^(1)
    diag = {(2, 0): S2_E2, (4, 0): _S4, (3, 1): DIAG_E31}
    for klucz in E:
        E[klucz] = xp.where(eye, diag.get(klucz, 0.0), E[klucz])
    return E


def suma_strukturalna(E, n, j, l, N, xp):
    """e_{n1..np}^{(j1..jp)(l1..lp)} = N^{-(alpha/2+1)} sum_{k0..kp} prod_s C^{l_s} E_{n_s}^{(j_s)}(a_{k_{s-1}} - a_{k_s}),
    alpha = sum (n_s - j_s). Liczone jako 1^T M_1 ... M_p 1 (p mnożeń macierz x wektor)."""
    v = xp.ones(N, dtype=xp.complex128)
    for ns, js, ls in zip(reversed(n), reversed(j), reversed(l)):
        M = E[(ns, js)]
        v = (xp.conj(M) if ls else M) @ v
    alpha = sum(n) - sum(j)
    return complex(xp.sum(v)) / N ** (alpha / 2.0 + 1.0)


# ---------------------------------------------------------------------------------------------------------------
# WZORY (41), (42), (43) — wygenerowane z elasticity2.tex
# ---------------------------------------------------------------------------------------------------------------
def _A4_wzor(e, RE, r1, r2, r3):
    """A4 = wzór (41) z Czapla, Drygaś, Gluzman, Ligocki, Mityushev, Materials 2025, 18, 5041 (wygenerowany automatycznie z elasticity2.tex).
    e(n, j, l) = suma strukturalna e_n^(j)(l);  RE = część rzeczywista."""
    return (
        ((1)/(_PI**3))*(-2*r1*r2*r3*e((3,3),(0,0),(1,0))
            -12*r3**3*e((4,3),(0,1),(1,0))
            -12*r3**3*e((3,4),(1,0),(1,0))
            -18*r3**3*e((4,4),(1,1),(1,0))
            -8*r3**4*e((3,3,3),(1,1,1),(1,0,1))
            -2*r3**2*((r3-r1)/(1+r1))*(e((2,2,3),(0,0,1),(0,0,1))
              +e((2,2,3),(0,0,1),(1,1,0))
              +2*e((2,2,3),(0,0,1),(1,0,1))))
    )


def _A5_wzor(e, RE, r1, r2, r3):
    """A5 = wzór (42) z Czapla, Drygaś, Gluzman, Ligocki, Mityushev, Materials 2025, 18, 5041 (wygenerowany automatycznie z elasticity2.tex).
    e(n, j, l) = suma strukturalna e_n^(j)(l);  RE = część rzeczywista."""
    return (
        ((1)/(_PI**4))*(3*r3*(r1*r2
              +12*r3**2)*e((4,4),(0,0),(1,0))
            +24*r3**3*(5*RE(e((5,4),(0,1),(1,0)))
              +2*e((5,5),(1,1),(1,0)))
            +8*r1*r2*r3**2*e((3,3,3),(0,0,1),(1,0,1))
            +16*r3**5*e((3,3,3,3),(1,1,1,1),(1,0,1,0))
            +24*r3**4*(2*e((4,3,3),(0,1,1),(1,0,1))
              +e((3,4,3),(1,0,1),(1,0,1))
              +3*e((4,4,3),(1,1,1),(1,0,1)))
            +12*r3**2*((r1-r3)/(1+r1))*(e((2,2,4),(0,0,0),(1,0,1))
              +e((2,3,4),(0,0,1),(1,0,1))
              -RE(e((2,2,4),(0,0,0),(1,1,0))+e((2,3,4),(0,0,1),(1,1,0))))
            +r3*(((r1-r3)/(1+r1)))**2*(e((2,2,2,2),(0,0,0,0),(1,0,1,0))
              -2*e((2,2,2,2),(0,0,0,0),(1,0,1,1))
              +e((2,2,2,2),(0,0,0,0),(1,1,0,0)))
            +4*r3**3*((r1-r3)/(1+r1))*(2*RE(e((2,2,3,3),(0,0,1,1),(1,0,1,0)))
              -2*e((2,2,3,3),(0,0,1,1),(1,1,0,1))
              -e((3,2,2,3),(1,0,0,1),(1,0,0,1))
              +e((3,2,2,3),(1,0,0,1),(1,0,1,0))))
    )


def _A6_wzor(e, RE, r1, r2, r3):
    """A6 = wzór (43) z Czapla, Drygaś, Gluzman, Ligocki, Mityushev, Materials 2025, 18, 5041 (wygenerowany automatycznie z elasticity2.tex).
    e(n, j, l) = suma strukturalna e_n^(j)(l);  RE = część rzeczywista."""
    return (
        ((1)/(_PI**5))*(-4*r3*(r1*r2
              +50*r3**2)*e((5,5),(0,0),(1,0))
            -r3**3*(360*RE(e((6,5),(0,1),(1,0)))
              +e((6,6),(1,1),(1,0)))
            +r3**2*((r1-r3)/(1+r1))*(-40*e((2,3,5),(0,0,0),(1,0,1))
              +40*RE(e((2,3,5),(0,0,0),(1,1,0)))
              -24*e((2,4,5),(0,0,1),(1,0,1))
              +24*RE(e((2,4,5),(0,0,1),(1,1,0))))
            +r1*r2*r3**2*(-24*e((3,3,4),(0,0,0),(1,0,1))
              -36*e((3,4,4),(0,0,1),(1,0,1)))
            +12*r3**2*(r1*r2
              +12*r3**2)*e((4,4,3),(0,0,1),(1,0,1))
            +r3**4*(-72*e((4,3,4),(0,1,0),(1,0,1))
              -216*e((4,4,4),(0,1,1),(1,0,1))
              -240*e((5,4,3),(0,1,1),(1,0,1))
              -240*e((3,5,4),(1,0,1),(1,0,1))
              -192*e((3,5,5),(1,1,1),(1,0,1))
              -216*e((4,5,4),(1,1,1),(1,0,1)))
            -4*r1*r2*r3*((r1-r3)/(1+r1))*(RE(e((2,2,3,3),(0,0,0,0),(1,0,1,0)))
              -e((2,2,3,3),(0,0,0,0),(1,1,0,1)))
            +12*r3**3*((r1-r3)/(1+r1))*(RE(-2*e((2,2,4,3),(0,0,0,1),(1,0,1,0))-2*e((2,2,3,4),(0,0,1,0),(1,0,1,0))-3*e((2,2,4,4),(0,0,1,1),(1,0,1,0))-2*e((2,3,4,3),(0,0,1,1),(1,0,1,0))-2*e((4,2,2,3),(0,0,0,1),(1,0,1,0))-2*e((3,2,3,4),(1,0,0,1),(1,0,1,0)))
              +2*e((2,2,4,3),(0,0,0,1),(1,1,0,1))
              +2*e((2,2,3,4),(0,0,1,0),(1,1,0,1))
              +3*e((2,2,4,4),(0,0,1,1),(1,1,0,1))
              +2*e((2,3,4,3),(0,0,1,1),(1,1,0,1))
              +2*e((4,2,2,3),(0,0,0,1),(1,0,0,1))
              +2*e((3,2,3,4),(1,0,0,1),(1,0,0,1)))
            -2*r3*(((r1-r3)/(1+r1)))**2*(e((2,3,3,2),(0,0,0,0),(1,0,1,0))
              +e((2,3,3,2),(0,0,0,0),(1,1,0,0))
              -e((2,3,3,2),(0,0,0,0),(1,1,0,1)))
            -8*r1*r2*r3**3*(2*RE(e((3,3,3,3),(0,0,1,1),(1,0,1,0)))
              +e((3,3,3,3),(1,0,0,1),(1,0,1,0)))
            -24*r3**5*(4*e((4,3,3,3),(0,1,1,1),(1,0,1,0))
              +3*e((3,4,4,3),(1,1,1,1),(1,0,1,0))
              +RE(4*e((3,4,3,3),(1,0,1,1),(1,0,1,0))+6*e((3,3,4,4),(1,1,1,1),(1,0,1,0))))
            +2*r3**2*(((r1-r3)/(1+r1)))**2*(2*RE(e((2,2,2,2,3),(0,0,0,0,1),(1,0,1,1,0))+e((2,2,3,2,2),(0,0,1,0,0),(1,0,1,0,0))+e((2,2,2,2,3),(0,0,0,0,1),(1,1,0,1,0)))
              -2*e((2,2,2,2,3),(0,0,0,0,1),(1,0,1,0,1))
              -e((2,2,3,2,2),(0,0,1,0,0),(1,0,1,0,1))
              -e((2,2,3,2,2),(0,0,1,0,0),(1,1,0,1,1))
              -e((3,2,2,2,2),(1,0,0,0,0),(1,0,0,1,1)))
            +16*r3**4*((r1-r3)/(1+r1))*(RE(e((2,2,3,3,3),(0,0,1,1,1),(1,1,0,1,0))+e((3,2,2,3,3),(1,0,0,1,1),(1,0,0,1,0)))
              -e((3,2,2,3,3),(1,0,0,1,1),(1,0,1,0,1))
              -e((3,3,3,2,2),(1,1,1,0,0),(1,0,1,0,1)))
            -32*r3**6*e((3,3,3,3,3),(1,1,1,1,1),(1,0,1,0,1)))
    )

def _RE(z):
    return complex(z).real


def wspolczynniki_A456(a, xp, rho1=None, rho2=None, rho3=None):
    """Zwraca dict zespolonych A4, A5, A6 dla położeń a (1-D, zespolone, kwadrat jednostkowy)."""
    rho1 = RHO1 if rho1 is None else rho1
    rho2 = RHO2 if rho2 is None else rho2
    rho3 = RHO3 if rho3 is None else rho3
    N = int(a.shape[0])
    E = macierze_E(a, xp)
    pomin_E2 = (rho1 == rho3)           # wszystkie sumy z n=2 mnożą się przez (rho1 - rho3): wtedy je pomijamy
    cache = {}

    def e(n, j, l):
        klucz = (n, j, l)
        if klucz not in cache:
            cache[klucz] = 0j if (pomin_E2 and 2 in n) else suma_strukturalna(E, n, j, l, N, xp)
        return cache[klucz]

    return dict(A4=_A4_wzor(e, _RE, rho1, rho2, rho3),
                A5=_A5_wzor(e, _RE, rho1, rho2, rho3),
                A6=_A6_wzor(e, _RE, rho1, rho2, rho3))


# ---------------------------------------------------------------------------------------------------------------
# lepkosc_okna: wynik v16 + A4, A5, A6 (zastępuje funkcję z komórki 3d; prepare() wywołuje ją po nazwie)
# ---------------------------------------------------------------------------------------------------------------
try:
    _lepkosc_okna_v16
except NameError:
    _lepkosc_okna_v16 = lepkosc_okna      # oryginał z komórki 3d (zachowany przy ponownym uruchomieniu tej komórki)


def lepkosc_okna(a, xp):
    """Jak v16 (e33, e4, A3, mu_e/mu rzędu f^3) + Re/Im A4, A5, A6 oraz mu_e/mu z szeregiem do f^4, f^5, f^6."""
    wynik = _lepkosc_okna_v16(a, xp)
    if wynik is None:
        return None
    nan = float('nan')
    dodatki = dict(Re_A4=nan, Im_A4=nan, Re_A5=nan, Im_A5=nan, Re_A6=nan, Im_A6=nan,
                   mu_e_A4=nan, mu_e_A5=nan, mu_e_A6=nan)
    if LICZ_A456:
        try:
            A = wspolczynniki_A456(a, xp)
            f = wynik['f']
            s = f + wynik['Re_A3'] * f ** 3                    # = s z v16 (A1 = RHO3 = 1, A2 = 0)
            for k in (4, 5, 6):
                Ak = A[f'A{k}']
                dodatki[f'Re_A{k}'], dodatki[f'Im_A{k}'] = Ak.real, Ak.imag
                s = s + Ak.real * f ** k
                dodatki[f'mu_e_A{k}'] = (1.0 + s) / (1.0 - KAPPA * s) if KAPPA * s < 1.0 else float('nan')
        except Exception as exc:
            print(f"[A456] błąd obliczeń A4-A6 (N={wynik.get('N')}): {exc}")
    wynik.update(dodatki)
    return wynik


# ---------------------------------------------------------------------------------------------------------------
# ZAPIS TEKSTOWY wyników okien klatki (wywoływany z przetworz_klatke po zapisie real{i}_lepkosc_okna.csv)
# ---------------------------------------------------------------------------------------------------------------
KOLUMNY_OKNA_TXT = ['i', 'nr', 'mr', 'y0', 'x0', 'N', 'f', 'e33', 'Re_e4', 'Im_e4', 'Re_A3', 'Im_A3',
                    'Re_A4', 'Im_A4', 'Re_A5', 'Im_A5', 'Re_A6', 'Im_A6',
                    'mu_e_pade', 'mu_e_A4', 'mu_e_A5', 'mu_e_A6', 'mu_e_wz15', 'mu_e_szereg']


def zapisz_okna_txt(i):
    """real{i}_lepkosc_okna.txt: jeden wiersz = jedno okno pływające klatki i (kolumny rozdzielone spacjami)."""
    if not WYNIKI_LEPKOSC:
        return None
    tab = pd.DataFrame(WYNIKI_LEPKOSC)
    kol = [c for c in KOLUMNY_OKNA_TXT if c in tab.columns]
    tab = tab.sort_values(['nr', 'mr'])[kol]
    plik = PATH_TXT + f'real{i}_lepkosc_okna.txt'
    with open(plik, 'w', encoding='utf-8') as f:
        f.write(f"# Klatka i={i}: wyniki z okien pływających ({len(tab)} okien)\n")
        f.write(f"# r_px={R_BAKT}, okno {x_d}x{y_d} px, rho1=rho2=rho3={RHO3}, kappa={KAPPA}, S2={S2_E2}, "
                f"E3^(1)(0)={DIAG_E31}\n")
        f.write("# A4, A5, A6: wzory (41)-(43) Part II (Materials 2025, 18, 5041); mu_e_Ak: (1+s)/(1-kappa s), "
                "s = f + ReA3 f^3 + ... + ReAk f^k\n")
        f.write("# " + " ".join(f"{c:>14s}" for c in kol) + "\n")
        for _, w in tab.iterrows():
            f.write("  " + " ".join(
                f"{int(w[c]):>14d}" if c in ('i', 'nr', 'mr', 'y0', 'x0', 'N') else f"{float(w[c]):>14.6e}"
                for c in kol) + "\n")
    print(f"[A456] Zapisano: {plik}")
    return plik


# ---------------------------------------------------------------------------------------------------------------
# TEST SAMOKONTROLNY (kilka sekund; drukuje PASS/FAIL, niczego nie zapisuje)
# ---------------------------------------------------------------------------------------------------------------
def _selftest_A456(verbose=True):
    import itertools
    rng = np.random.default_rng(12345)
    wyniki = []

    def ok(nazwa, warunek, info=''):
        wyniki.append(bool(warunek))
        if verbose:
            print(f"  [{'PASS' if warunek else 'FAIL'}] {nazwa} {info}")

    # 1. E_n^(1) i E_n z wzorów zamkniętych = bezpośrednie sumy sieciowe (n >= 4: szereg zbieżny bezwzględnie)
    z0 = np.array([0.23 + 0.11j, -0.31 + 0.40j])
    E = macierze_E(np.array([0.0, 0.23 + 0.11j, -0.31 + 0.40j]), np)
    m = np.arange(-120, 121)
    P = (m[:, None] + 1j * m[None, :]).ravel()
    for (n, j) in [(4, 0), (5, 0), (6, 0), (4, 1), (5, 1), (6, 1)]:
        blad = 0.0
        for k, zz in enumerate(z0):
            w = zz - P
            bezp = np.sum(np.conj(w) ** j / w ** n)
            blad = max(blad, abs(E[(n, j)][k + 1, 0] - bezp) / max(1.0, abs(bezp)))
        ok(f"E_{n}^({j}) = suma sieciowa", blad < 1e-3, f"(błąd wzgl. {blad:.1e})")

    # 2. algorytm łańcuchowy = sumowanie wprost (p = 3, mały N)
    N = 5
    a = rng.random(N) + 1j * rng.random(N)
    E = macierze_E(a, np)
    klucz = ((3, 4, 3), (1, 0, 1), (1, 0, 1))
    dd = a[:, None] - a[None, :]
    wprost = 0j
    for k in itertools.product(range(N), repeat=4):
        t = 1 + 0j
        for s_, (ns, js, ls) in enumerate(zip(*klucz)):
            v = E[(ns, js)][k[s_], k[s_ + 1]]
            t *= np.conj(v) if ls else v
        wprost += t
    wprost /= N ** ((sum(klucz[0]) - sum(klucz[1])) / 2.0 + 1.0)
    lanc = suma_strukturalna(E, *klucz, N, np)
    ok("łańcuch macierzowy = suma po wszystkich krotkach", abs(lanc - wprost) < 1e-9 * max(1, abs(wprost)),
       f"({abs(lanc - wprost):.1e})")

    # 3. zgodność z e33, e4 z komórki 3d
    N = 40
    a = rng.random(N) + 1j * rng.random(N)
    E = macierze_E(a, np)
    e33_ref, e4r_ref, e4i_ref = sumy_e33_e4(a, np)
    e33 = suma_strukturalna(E, (3, 3), (1, 1), (1, 0), N, np)
    e4 = suma_strukturalna(E, (4,), (0,), (1,), N, np)
    ok("e33 = sumy_e33_e4 (v16)", abs(e33 - e33_ref) < 1e-9 * abs(e33_ref), f"({e33:.6g} vs {e33_ref:.6g})")
    ok("e4  = sumy_e33_e4 (v16)", abs(e4 - complex(e4r_ref, e4i_ref)) < 1e-9 * abs(e4))

    # 4. niezmienniczość względem przesunięcia wszystkich punktów (okresowość E_n^(j) i poprawność sprzężeń)
    A0 = wspolczynniki_A456(a, np)
    przes = a + (0.37 + 0.21j)
    przes = (przes.real % 1.0) + 1j * (przes.imag % 1.0)
    A1 = wspolczynniki_A456(przes, np)
    for k in ('A4', 'A5', 'A6'):
        ok(f"{k} niezmiennicze względem przesunięcia", abs(A0[k] - A1[k]) < 1e-7 * max(1.0, abs(A0[k])),
           f"({A0[k]:.6g})")

    # 5. dla rho1 = rho3 pominięcie sum z E2 nie zmienia wyniku
    Ad = wspolczynniki_A456(a, np, 0.8, 0.5, 0.8)
    E = macierze_E(a, np)
    N = len(a)
    cache = {}

    def e_pelne(n, j, l):
        if (n, j, l) not in cache:
            cache[(n, j, l)] = suma_strukturalna(E, n, j, l, N, np)
        return cache[(n, j, l)]
    ok("pominięcie E2 dla rho1 = rho3", abs(_A6_wzor(e_pelne, _RE, 0.8, 0.5, 0.8) - Ad['A6']) < 1e-9 * abs(Ad['A6']))
    if verbose:
        print("  PASS" if all(wyniki) else "  UWAGA: są błędy!", f"({sum(wyniki)}/{len(wyniki)} testów)")
    return all(wyniki)


_selftest_A456()


In [ ]:
# Komórka 4: pętla główna po klatkach

t_global_start = time.time()
wyniki = []

for i in range(i_min, i_max + 1):
    try:
        wynik = przetworz_klatke(i)
        wyniki.append(wynik)
        gpu_mem_info(f"po i={i}")
    except Exception as e:
        import traceback
        print(f"\n!!! KRYTYCZNY BŁĄD dla i={i}: {e}")
        traceback.print_exc()
        wyniki.append({
            'i': i, 'txt': False, 'csv': False,
            'png_3d': 0, 'png_2d': 0,
            'stat': False, 'czas': 0
        })
        try:
            import cupy as cp
            cp.get_default_memory_pool().free_all_blocks()
        except Exception:
            pass
        try:
            _torus_cache.pop(i, None)
            _blobs_cache.pop(i, None)
        except Exception:
            pass

t_global_end = time.time()
print(f"\n{'='*70}")
print(f"PODSUMOWANIE KOŃCOWE")
print(f"{'='*70}")

n_ok = sum(1 for w in wyniki if w['txt'])
n_csv = sum(1 for w in wyniki if w['csv'])
n_stat = sum(1 for w in wyniki if w.get('stat', False))
n_3d_total = sum(w['png_3d'] for w in wyniki)
n_2d_total = sum(w['png_2d'] for w in wyniki)
czas_total = t_global_end - t_global_start

print(f"Klatek przetworzonych: {len(wyniki)} (z {i_max - i_min + 1})")
print(f"  z TXT:               {n_ok}")
print(f"  z CSV:               {n_csv}")
print(f"  z resultN.txt:       {n_stat}")
print(f"PNG 3D:                {n_3d_total}")
print(f"PNG 2D:                {n_2d_total}")
print(f"Łączny czas:           {czas_total:.1f} s ({czas_total/60:.1f} min)")

if wyniki:
    czasy = [w['czas'] for w in wyniki if w['czas'] > 0]
    if czasy:
        print(f"Średni czas na klatkę: {sum(czasy)/len(czasy):.1f} s")

pominiete = [w['i'] for w in wyniki if not w['txt']]
if pominiete:
    print(f"\nKlatek bez TXT: {len(pominiete)}")
    if len(pominiete) <= 20:
        print(f"  Numery: {pominiete}")
    else:
        print(f"  Numery: {pominiete[:10]} ... {pominiete[-10:]}")

try:
    _torus_cache.clear()
    _blobs_cache.clear()
    print(f"\n[CACHE] Wyczyszczono cache torusa i blobów.")
except Exception:
    pass

print(f"\nKatalogi wynikowe:")
try:
    print(f"  TXT: {PATH_TXT}")
    print(f"  PNG: {PATH_PNG}")
    print(f"  CSV: {PATH_CSV}")
except NameError:
    print(f"  (brak zmiennych PATH_* — uruchom Komórkę 1)")

print(f"{'='*70}")

In [ ]:
# Komórka 5: zbiorczy result_all.txt

import os
import numpy as np
import pandas as pd

try:
    _i_min, _i_max = i_min, i_max
except NameError:
    _i_min, _i_max = 1, 239
try:
    COLUMNS_CSV
except NameError:
    COLUMNS_CSV = ["N", "y", "x", "Re(e2)", "Im(e2)", "|e2|", "e22"]
try:
    WIELKOSCI_STAT
except NameError:
    WIELKOSCI_STAT = ["Re(e2)", "Im(e2)", "|e2|", "e22"]

try:
    _path_txt = PATH_TXT
    _path_csv = PATH_CSV
except NameError:
    _path = '/content/drive/MyDrive/'
    _path_txt = _path + 'TXT/'
    _path_csv = _path + 'CSV/'
    os.makedirs(_path_txt, exist_ok=True)
    os.makedirs(_path_csv, exist_ok=True)


def wczytaj_csv_stat(csv_file):
    if not os.path.exists(csv_file):
        return None
    try:
        df = pd.read_csv(csv_file, delimiter=",", decimal=".",
                         comment="#", header=0, encoding='utf-8')
        if len(df.columns) == len(COLUMNS_CSV):
            df.columns = COLUMNS_CSV
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        potrzebne = ["x", "y"] + WIELKOSCI_STAT
        for c in potrzebne:
            if c not in df.columns:
                return None
        df = df.dropna(subset=potrzebne)
        if len(df) < 1:
            return None
        return df
    except Exception as e:
        print(f"[WARN] Błąd wczytywania {csv_file}: {e}")
        return None


def statystyki(df):
    wynik = {}
    for c in WIELKOSCI_STAT:
        v = df[c].to_numpy(dtype=np.float64)
        wynik[c] = {
            'mean':   float(np.mean(v)),
            'std':    float(np.std(v, ddof=0)),
            'median': float(np.median(v)),
            'q25':    float(np.percentile(v, 25)),
            'q75':    float(np.percentile(v, 75)),
            'n':      int(len(v)),
        }
    return wynik


result_all = _path_txt + 'result_all.txt'

lacznie_plikow = 0
lacznie_wierszy = 0
pominiete = []
bledy = []

with open(result_all, 'w', encoding='utf-8') as f_all:
    f_all.write("# Zbiorcze statystyki dla wszystkich klatek\n")
    f_all.write(f"# Zakres klatek: i ∈ [{_i_min}, {_i_max}]\n")
    f_all.write("# Format: i  N  "
                "Re(e2)_mean  Re(e2)_median  Re(e2)_std  "
                "Im(e2)_mean  Im(e2)_median  Im(e2)_std  "
                "|e2|_mean  |e2|_median  |e2|_std  "
                "e22_mean  e22_median  e22_std\n")
    f_all.write("#" + "-" * 160 + "\n")

    for i in range(_i_min, _i_max + 1):
        csv_file = _path_csv + f'real{i}.csv'
        df = wczytaj_csv_stat(csv_file)
        if df is None:
            pominiete.append(i)
            continue

        try:
            stats = statystyki(df)
            n_wierszy = len(df)
            f_all.write(
                f"{i:>4d}  {n_wierszy:>4d}  "
                f"{stats['Re(e2)']['mean']:> .4e}  "
                f"{stats['Re(e2)']['median']:> .4e}  "
                f"{stats['Re(e2)']['std']:> .4e}  "
                f"{stats['Im(e2)']['mean']:> .4e}  "
                f"{stats['Im(e2)']['median']:> .4e}  "
                f"{stats['Im(e2)']['std']:> .4e}  "
                f"{stats['|e2|']['mean']:> .4e}  "
                f"{stats['|e2|']['median']:> .4e}  "
                f"{stats['|e2|']['std']:> .4e}  "
                f"{stats['e22']['mean']:> .4e}  "
                f"{stats['e22']['median']:> .4e}  "
                f"{stats['e22']['std']:> .4e}\n"
            )
            lacznie_plikow += 1
            lacznie_wierszy += n_wierszy
            print(f"[OK] real{i}.csv → result_all.txt  "
                  f"(N={n_wierszy}, e22: mean={stats['e22']['mean']:.4e}, "
                  f"median={stats['e22']['median']:.4e})")
        except Exception as e:
            print(f"[BŁĄD] real{i}.csv: {e}")
            bledy.append((i, str(e)))

print()
print(f"=== Podsumowanie Komórki 5 ===")
print(f"Przetworzonych plików:     {lacznie_plikow}")
print(f"Łącznie wierszy danych:    {lacznie_wierszy}")
print(f"Zbiorczy plik:             {result_all}")
if pominiete:
    print(f"Pominięto (brak CSV):      {len(pominiete)}")
    if len(pominiete) <= 20:
        print(f"  Numery: {pominiete}")
    else:
        print(f"  Numery: {pominiete[:10]} ... {pominiete[-10:]}")
if bledy:
    print(f"Błędy:                     {len(bledy)}")
    for i, err in bledy[:5]:
        print(f"  i={i}: {err}")


# =====================================================================
# RAPORT STATYSTYK LEPKOŚCI  ->  result_lepkosc_all.txt  (+ result_lepkosc_all.csv)
# Źródło: real{i}_lepkosc_okna.csv (wszystkie okna klatki)
#         real{i}_lepkosc.csv      (wybrane miejsca H1,H2,L1,L2)
# =====================================================================
LEP_KOLUMNY = ['f', 'e33', 'Re_e4', 'Im_e4', 'Re_A3', 'Im_A3',
               'Re_A4', 'Im_A4', 'Re_A5', 'Im_A5', 'Re_A6', 'Im_A6',        # v17 (komórka 3e)
               'mu_e_pade', 'mu_e_A4', 'mu_e_A5', 'mu_e_A6', 'mu_e_wz15', 'mu_e_szereg']

lep_report = _path_txt + 'result_lepkosc_all.txt'
lep_csv_zbiorczy = _path_csv + 'result_lepkosc_all.csv'


def _stat_lep(v):
    v = np.asarray(v, dtype=np.float64)
    n_all = len(v)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return dict(mean=np.nan, std=np.nan, min=np.nan, q25=np.nan, median=np.nan,
                    q75=np.nan, max=np.nan, n=0, n_nan=n_all)
    return dict(mean=float(np.mean(v)), std=float(np.std(v, ddof=0)), min=float(np.min(v)),
                q25=float(np.percentile(v, 25)), median=float(np.median(v)),
                q75=float(np.percentile(v, 75)), max=float(np.max(v)),
                n=int(len(v)), n_nan=int(n_all - len(v)))


wiersze_csv = []
przetw_lep, pomin_lep = 0, []

with open(lep_report, 'w', encoding='utf-8') as fr:
    fr.write("# RAPORT STATYSTYK LEPKOŚCI EFEKTYWNEJ\n")
    fr.write(f"# Zakres klatek: i ∈ [{_i_min}, {_i_max}]\n")
    try:
        fr.write(f"# r_px (R_BAKT) = {R_BAKT} px, okno {x_d}x{y_d} px, MIN_N_OKNA = {MIN_N_OKNA}\n")
    except NameError:
        pass
    fr.write("# Wzory (F.tex): A3 = (2/pi^2)(2 e33 + 3 e4);  s = f + Re A3 f^3;  mu_e/mu = (1+s)/(1-s)\n")
    fr.write("# mu_e_pade = (1+s)/(1-s), s wg (14);  mu_e_wz15 = wzór (15);  mu_e_szereg = wzór (1) ECMS 2024\n")
    fr.write("# n_nan = liczba okien bez wartości (np. s >= 1 -> brak mu_e/mu)\n")
    fr.write("#" + "=" * 150 + "\n")

    for i in range(_i_min, _i_max + 1):
        f_okna = _path_csv + f'real{i}_lepkosc_okna.csv'
        f_hl = _path_csv + f'real{i}_lepkosc.csv'
        if not os.path.isfile(f_okna):
            pomin_lep.append(i)
            continue
        try:
            tab = pd.read_csv(f_okna)
        except Exception as e:
            print(f"[LEPK-RAPORT] i={i}: nieczytelny {f_okna} ({e})")
            pomin_lep.append(i)
            continue
        if tab.empty:
            pomin_lep.append(i)
            continue

        fr.write(f"\n### Klatka i={i}: okien z wynikiem = {len(tab)}\n")
        fr.write(f"{'wielkosc':>12s} {'mean':>11s} {'std':>11s} {'min':>11s} {'q25':>11s} "
                 f"{'median':>11s} {'q75':>11s} {'max':>11s} {'n':>5s} {'n_nan':>6s}\n")
        for col in LEP_KOLUMNY:
            if col not in tab.columns:
                continue
            st = _stat_lep(tab[col].to_numpy())
            fr.write(f"{col:>12s} {st['mean']:>11.4e} {st['std']:>11.4e} {st['min']:>11.4e} "
                     f"{st['q25']:>11.4e} {st['median']:>11.4e} {st['q75']:>11.4e} "
                     f"{st['max']:>11.4e} {st['n']:>5d} {st['n_nan']:>6d}\n")
            wiersze_csv.append(dict(i=i, wielkosc=col, **st))

        # wybrane miejsca H1, H2, L1, L2
        if os.path.isfile(f_hl):
            try:
                hl = pd.read_csv(f_hl)
                kol = [c_ for c_ in ['miejsce', 'x_okna', 'y_okna', 'N', 'f',
                                     'Re_A3', 'Im_A3', 'mu_e_przez_mu']
                       if c_ in hl.columns]
                fr.write(f"\n  Wybrane miejsca (H = wysokie N, L = niskie N), klatka i={i}:\n")
                for linia in hl[kol].to_string(index=False, float_format=lambda v: f"{v:.5g}").splitlines():
                    fr.write("  " + linia + "\n")
            except Exception as e:
                fr.write(f"  # błąd odczytu {f_hl}: {e}\n")
        przetw_lep += 1

    # zbiorcze statystyki po wszystkich oknach wszystkich klatek
    fr.write("\n" + "#" + "=" * 150 + "\n")
    fr.write("### ZBIORCZO: wszystkie okna wszystkich klatek\n")
    wszystkie = []
    for i in range(_i_min, _i_max + 1):
        f_okna = _path_csv + f'real{i}_lepkosc_okna.csv'
        if os.path.isfile(f_okna):
            try:
                t_ = pd.read_csv(f_okna)
                if not t_.empty:
                    wszystkie.append(t_)
            except Exception:
                pass
    if wszystkie:
        tab_all = pd.concat(wszystkie, ignore_index=True)
        fr.write(f"# okien łącznie: {len(tab_all)}\n")
        fr.write(f"{'wielkosc':>12s} {'mean':>11s} {'std':>11s} {'min':>11s} {'q25':>11s} "
                 f"{'median':>11s} {'q75':>11s} {'max':>11s} {'n':>6s} {'n_nan':>6s}\n")
        for col in LEP_KOLUMNY:
            if col not in tab_all.columns:
                continue
            st = _stat_lep(tab_all[col].to_numpy())
            fr.write(f"{col:>12s} {st['mean']:>11.4e} {st['std']:>11.4e} {st['min']:>11.4e} "
                     f"{st['q25']:>11.4e} {st['median']:>11.4e} {st['q75']:>11.4e} "
                     f"{st['max']:>11.4e} {st['n']:>6d} {st['n_nan']:>6d}\n")
            wiersze_csv.append(dict(i='ALL', wielkosc=col, **st))
    else:
        fr.write("# brak danych (uruchom komórki 3b, 3d i 4)\n")

if wiersze_csv:
    pd.DataFrame(wiersze_csv).to_csv(lep_csv_zbiorczy, index=False, float_format='%.6g')

print()
print(f"=== Raport lepkości ===")
print(f"Klatek w raporcie:         {przetw_lep}")
print(f"Raport tekstowy:           {lep_report}")
print(f"Raport CSV:                {lep_csv_zbiorczy if wiersze_csv else '(brak danych)'}")
if pomin_lep:
    print(f"Pominięto (brak danych lepkości): {len(pomin_lep)}")

In [ ]:
# Komórka 5b (v17): SZEREGI CZASOWE lepkości i współczynników A3-A6 po klatkach (uruchom PO pętli głównej i komórce 5)
# Źródło: real{i}_lepkosc_okna.csv (wszystkie okna klatki i; kolumny dodaje komórka 3e).
# Wynik:
#   result_A456_okna_szeregi.csv   – wszystkie okna wszystkich klatek, posortowane (nr, mr, i): okno o stałych (nr, mr)
#                                    leży w tym samym miejscu każdej klatki, więc wiersze kolejnych i to szereg czasowy okna
#   result_A456_klatki_szereg.csv  – jeden wiersz = jedna klatka: statystyki po oknach (mean, std, median, q25, q75, min,
#                                    max, n, n_nan) każdej wielkości = szereg czasowy statystyk klatek
#   result_A456_klatki_szereg.txt  – to samo w skrócie (mean, median, std), kolumny rozdzielone spacjami
import os
import numpy as np
import pandas as pd

try:
    _i_min, _i_max = i_min, i_max
except NameError:
    _i_min, _i_max = 1, 239
try:
    _p_csv, _p_txt = PATH_CSV, PATH_TXT
except NameError:
    _p_csv = _p_txt = './'

FPS = 25.0   # Movie S1.AVI: 25 kl./s (239 klatek = 9,56 s); t_s = (i - 1) / FPS — ustaw inaczej, jeśli klatki były rzadsze
KOL_SZEREG = ['N', 'f', 'e33', 'Re_e4', 'Im_e4', 'Re_A3', 'Im_A3', 'Re_A4', 'Im_A4', 'Re_A5', 'Im_A5',
              'Re_A6', 'Im_A6', 'mu_e_pade', 'mu_e_A4', 'mu_e_A5', 'mu_e_A6']


def _stat_wektora(v):
    v = np.asarray(v, dtype=np.float64)
    n_all = len(v)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return dict(mean=np.nan, std=np.nan, median=np.nan, q25=np.nan, q75=np.nan, min=np.nan, max=np.nan,
                    n=0, n_nan=n_all)
    return dict(mean=float(np.mean(v)), std=float(np.std(v, ddof=0)), median=float(np.median(v)),
                q25=float(np.percentile(v, 25)), q75=float(np.percentile(v, 75)),
                min=float(np.min(v)), max=float(np.max(v)), n=int(len(v)), n_nan=int(n_all - len(v)))


tabele, brak = [], []
for i_ in range(_i_min, _i_max + 1):
    plik = _p_csv + f'real{i_}_lepkosc_okna.csv'
    if not os.path.isfile(plik):
        brak.append(i_)
        continue
    try:
        t_ = pd.read_csv(plik)
    except Exception as exc:
        print(f"[SZEREG] i={i_}: nieczytelny {plik} ({exc})")
        brak.append(i_)
        continue
    if t_.empty:
        brak.append(i_)
        continue
    t_['i'] = i_
    tabele.append(t_)

if not tabele:
    print("[SZEREG] brak danych (uruchom komórki 3b, 3d, 3e i 4)")
else:
    tab_all = pd.concat(tabele, ignore_index=True)
    tab_all.insert(1, 't_s', (tab_all['i'] - 1) / FPS)
    if 'Re_A4' not in tab_all.columns:
        print("[SZEREG] UWAGA: w danych brak kolumn A4-A6 — klatki liczono bez komórki 3e (uruchom 3e i pętlę główną)")

    # (a) szeregi czasowe poszczególnych okien
    plik_okna = _p_csv + 'result_A456_okna_szeregi.csv'
    tab_all.sort_values(['nr', 'mr', 'i']).to_csv(plik_okna, index=False, float_format='%.8g')

    # (b) szeregi czasowe statystyk klatek
    wiersze = []
    for i_, g_ in tab_all.groupby('i'):
        w_ = dict(i=int(i_), t_s=(int(i_) - 1) / FPS, n_okien=int(len(g_)))
        for kol in KOL_SZEREG:
            if kol not in g_.columns:
                continue
            for nazwa, wartosc in _stat_wektora(g_[kol].to_numpy()).items():
                w_[f'{kol}_{nazwa}'] = wartosc
        wiersze.append(w_)
    szereg = pd.DataFrame(wiersze).sort_values('i')
    plik_klatki = _p_csv + 'result_A456_klatki_szereg.csv'
    szereg.to_csv(plik_klatki, index=False, float_format='%.8g')

    kol_txt = ['i', 't_s', 'n_okien'] + [f'{k}_{s}' for k in KOL_SZEREG for s in ('mean', 'median', 'std')
                                          if f'{k}_{s}' in szereg.columns]
    plik_txt = _p_txt + 'result_A456_klatki_szereg.txt'
    with open(plik_txt, 'w', encoding='utf-8') as f:
        f.write("# SZEREG CZASOWY STATYSTYK KLATEK: lepkość efektywna i współczynniki A3-A6 (statystyki po oknach pływających)\n")
        f.write(f"# Zakres klatek: i ∈ [{_i_min}, {_i_max}], t_s = (i-1)/{FPS:g} s; klatek z danymi: {len(szereg)}\n")
        f.write("# mu_e_pade = (1+s)/(1-s), s = f + ReA3 f^3;  mu_e_A4/A5/A6: ten sam wzór z dodatkowymi wyrazami ReA4 f^4, ReA5 f^5, ReA6 f^6\n")
        f.write("# " + " ".join(f"{c:>16s}" for c in kol_txt) + "\n")
        for _, w_ in szereg[kol_txt].iterrows():
            f.write("  " + " ".join(
                f"{int(w_[c]):>16d}" if c in ('i', 'n_okien') else f"{float(w_[c]):>16.8e}" for c in kol_txt) + "\n")

    print(f"=== Szeregi czasowe A3-A6 ===")
    print(f"Klatek z danymi:           {len(szereg)}  (brak: {len(brak)})")
    print(f"Okien łącznie:             {len(tab_all)}")
    print(f"CSV (okna, szeregi):       {plik_okna}")
    print(f"CSV (klatki, statystyki):  {plik_klatki}")
    print(f"TXT (klatki, skrót):       {plik_txt}")
    if brak:
        print(f"Brak danych dla klatek:    {brak if len(brak) <= 20 else str(brak[:10]) + ' ... ' + str(brak[-10:])}")


In [ ]:
# Komórka 6: anomalie f vs mu_e/mu — czy zdarza się większe f przy mniejszym mu_e/mu (lub odwrotnie)?
# Źródło: real{i}_lepkosc_okna.csv = WSZYSTKIE okna klatki (kolumny f, mu_e_pade, N, x0, y0).
# Wklej PO komórce 5 (potrzebuje: i_min, i_max, PATH_CSV, PATH_PNG, x_d, y_d).
#
# Definicja: para okien (A, B) z tej samej klatki jest DYSKORDANTNA, gdy f_A < f_B, a mu_A > mu_B (albo odwrotnie).
# Okna o identycznym f (ten sam N) pomijamy: f zależy tylko od N, więc przy tym samym N różnice mu wynikają
# wyłącznie z ułożenia bakterii (A3) i nie są "anomalią f–mu".
# score okna = udział okien o innym f, z którymi to okno tworzy parę dyskordantną (0 = zgodne ze wszystkimi).

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_OKIEN       = 20     # tyle najbardziej dyskordantnych okien na klatkę zapisujemy do CSV
TOP_KLATEK_MAPY = 5      # dla tylu klatek (największy udział dyskordantnych par) rysujemy mapę
EPS_F           = 1e-12  # tolerancja: f traktujemy jako równe, gdy |Δf| <= EPS_F

klatki, okna_top, mapy = [], [], {}
n_brak = n_okien = 0

for i in range(i_min, i_max + 1):
    fn = PATH_CSV + f'real{i}_lepkosc_okna.csv'
    if not os.path.isfile(fn):
        continue
    try:
        t = pd.read_csv(fn)
    except Exception as e:
        print(f"[ANOM] i={i}: nieczytelny {fn} ({e})")
        continue
    if t.empty or not {'f', 'mu_e_pade', 'x0', 'y0'} <= set(t.columns):
        continue
    n_okien += len(t)
    ok = np.isfinite(t['mu_e_pade'].to_numpy())
    n_brak += int((~ok).sum())
    tt = t[ok].reset_index(drop=True)
    if len(tt) < 3:
        continue

    f = tt['f'].to_numpy(dtype=np.float64)
    mu = tt['mu_e_pade'].to_numpy(dtype=np.float64)
    df_ = f[None, :] - f[:, None]                  # [a, b] = f_b - f_a
    dm_ = mu[None, :] - mu[:, None]                # [a, b] = mu_b - mu_a
    rozne = np.abs(df_) > EPS_F                    # pary o różnym f
    dysk = rozne & (df_ * dm_ < 0)                 # f rośnie, a mu maleje (lub odwrotnie)
    n_par, n_dysk = rozne.sum() / 2, dysk.sum() / 2
    tt['score'] = dysk.sum(axis=1) / np.maximum(rozne.sum(axis=1), 1)

    rho = tt[['f', 'mu_e_pade']].corr(method='spearman').iloc[0, 1]
    klatki.append(dict(i=i, okien=len(t), okien_bez_mu=int((~ok).sum()), par_o_roznym_f=int(n_par),
                       par_dyskordantnych=int(n_dysk), udzial_dyskordantnych=n_dysk / max(n_par, 1),
                       spearman_f_mu=rho, okien_score_ge_5proc=int((tt['score'] >= 0.05).sum())))
    kol = [c for c in ['i', 'x0', 'y0', 'N', 'f', 'mu_e_pade', 'Re_A3', 'Im_A3', 'score'] if c in tt.columns or c == 'i']
    top = tt.assign(i=i).sort_values('score', ascending=False).head(TOP_OKIEN)
    okna_top.append(top[kol])
    mapy[i] = (t, tt)

if not klatki:
    raise SystemExit("[ANOM] brak plików real{i}_lepkosc_okna.csv — najpierw uruchom pętlę główną (komórka 4).")

kl = pd.DataFrame(klatki)
top_df = pd.concat(okna_top, ignore_index=True)
kl.to_csv(PATH_CSV + 'result_lepkosc_anomalie_klatki.csv', index=False, float_format='%.6g')
top_df.to_csv(PATH_CSV + 'result_lepkosc_anomalie_okna.csv', index=False, float_format='%.6g')

print(f"[ANOM] klatek: {len(kl)}, okien: {n_okien}, bez mu_e/mu (s>=1, pominięte): {n_brak}")
print(f"[ANOM] udział par dyskordantnych (f↑ a mu↓): średnio {kl['udzial_dyskordantnych'].mean():.4f}, "
      f"max {kl['udzial_dyskordantnych'].max():.4f} (klatka i={int(kl.loc[kl['udzial_dyskordantnych'].idxmax(), 'i'])}); "
      f"Spearman(f, mu): średnio {kl['spearman_f_mu'].mean():.3f}")

# --- Mapy score dla klatek z największym udziałem dyskordantnych par ------------------------
for i in kl.sort_values('udzial_dyskordantnych', ascending=False)['i'].head(TOP_KLATEK_MAPY):
    t, tt = mapy[int(i)]
    xc, yc = tt['x0'] + x_d / 2, tt['y0'] + y_d / 2
    fig, ax = plt.subplots(figsize=(8, 7))
    brak = t[~np.isfinite(t['mu_e_pade'])]
    ax.scatter(brak['x0'] + x_d / 2, brak['y0'] + y_d / 2, c='lightgray', marker='x', s=10,
               label=f'brak mu_e/mu (s>=1): {len(brak)}')
    sc = ax.scatter(xc, yc, c=tt['score'], cmap='magma_r', marker='s', s=22, vmin=0)
    fig.colorbar(sc, ax=ax, label='score: udział okien dyskordantnych z danym oknem')
    top = tt.sort_values('score', ascending=False).head(5)
    ax.scatter(top['x0'] + x_d / 2, top['y0'] + y_d / 2, s=170, facecolors='none', edgecolors='cyan', linewidths=1.6)
    for r, (_, w) in enumerate(top.iterrows(), 1):
        ax.annotate(str(r), (w['x0'] + x_d / 2, w['y0'] + y_d / 2), xytext=(7, 7), textcoords='offset points',
                    color='deepskyblue', fontsize=9, weight='bold')
    ax.set_aspect('equal')
    ax.set_xlabel('x [px] (środek okna)'); ax.set_ylabel('y [px] (środek okna)')
    ax.set_title(f"Klatka i={int(i)}: niezgodność f i mu_e/mu\n(kółka 1-5 = najbardziej dyskordantne okna)")
    ax.legend(loc='upper right', fontsize=8)
    fn = PATH_PNG + f'real{int(i)}_2D_anomalie_lepkosc.png'
    fig.savefig(fn, dpi=100, bbox_inches='tight')
    plt.close(fig)
    print(f"[ANOM] Zapisano: {fn}")
print(f"[ANOM] Zapisano: {PATH_CSV}result_lepkosc_anomalie_klatki.csv, {PATH_CSV}result_lepkosc_anomalie_okna.csv")

In [ ]:
# Komórka 7: skrajne okna anomalii f–mu_e/mu i obrazki real{i}_2D_N_interp_anomalia.png
# Wklej PO komórce 6 (potrzebuje: komórek 3a–3d, PATH_CSV, PATH_PNG, x_d, y_d, GRID_N, INTERP_METHOD, CONC_LABEL oraz
# pliku result_lepkosc_anomalie_klatki.csv z komórki 6; pliki real{i}_lepkosc_okna.csv z pętli głównej).
#
# Dla każdej z klatek z największym udziałem anomalii szukamy ŁAŃCUCHA okien o kolejnych N (n0, n0+1, ..., n0+L-1),
# w którym f rośnie, a mu_e/mu ściśle MALEJE (każdy krok o co najmniej PRÓG). Wybieramy:
#   1) największy PRÓG z listy PROGI_KROKU, dla którego taki łańcuch istnieje,
#   2) wśród łańcuchów z tym progiem ten o największym łącznym spadku mu_e/mu.
# Skrajne okna: A = pierwsze (najmniejsze N, największe mu_e/mu), B = ostatnie (największe N, najmniejsze mu_e/mu).
# Rysujemy obrazek jak real{i}_2D_N_interp_lepkosc.png z zaznaczonymi oknami A i B oraz ich szóstką
# (x0, y0, N, f, Re_A3, mu_e/mu) w etykiecie każdego okna. Tylko dla tych klatek rysujemy też
# real{i}_3D_N_interp.png (wykres 3D interpolacji N).

import os
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

KLATKI_ANOMALII  = None    # None = LICZBA_KLATEK klatek o największym udziale par dyskordantnych (z komórki 6); lub np. [31, 233, 1, 105, 106]
LICZBA_KLATEK    = 5
DLUGOSC_LANCUCHA = 5       # ile okien o kolejnych N w łańcuchu (A = pierwsze, B = ostatnie); gdy brak, próbujemy krótszych
PROGI_KROKU      = [0.004, 0.003, 0.002, 0.001, 0.0005, 0.0001]   # minimalny spadek mu_e/mu między kolejnymi oknami


def _rysuj_3d_z_siatki(x, y, z, Xg, Yg, Zg, z_label, tytul):
    """Wykres 3D interpolacji N (4 widoki, VIEW_ANGLES) — rysowany tylko dla klatek z anomaliami."""
    vmin, vmax = CONC_VLIM if CONC_VLIM else (None, None)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), subplot_kw={'projection': '3d'})
    for ax, (elev, azim) in zip(axes.ravel(), VIEW_ANGLES):
        ax.scatter(x, y, z, c='red', s=10, depthshade=False)
        ax.plot_surface(Xg, Yg, Zg, cmap='viridis', alpha=0.8, vmin=vmin, vmax=vmax)
        ax.contour3D(Xg, Yg, Zg, 50, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel(z_label)
        ax.view_init(elev, azim)
        ax.set_title(f"{tytul}  |  Elev={elev}°, Azim={azim}°", fontsize=9)
    fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0.2, hspace=0.3)
    return fig


def _pl(x, nd):
    """Liczba z przecinkiem dziesiętnym."""
    return f"{x:.{nd}f}".replace('.', ',')


def znajdz_skrajne(tab, dlugosc=DLUGOSC_LANCUCHA, progi=PROGI_KROKU):
    """Zwraca (A, B, próg, łańcuch) albo None. A, B to wiersze tabeli okien (x0, y0, N, f, Re_A3, mu_e_pade, ...)."""
    t = tab[np.isfinite(tab['mu_e_pade'])]
    poziomy = {int(n): g.reset_index(drop=True) for n, g in t.groupby('N')}
    for L in range(dlugosc, 1, -1):
        for d in progi:
            best = None
            for n0 in sorted(poziomy):
                if any(n0 + k not in poziomy for k in range(L)):
                    continue
                lev = [poziomy[n0 + k] for k in range(L)]
                dp = np.zeros(len(lev[0]))
                prev = []
                for k in range(1, L):
                    krok = lev[k - 1]['mu_e_pade'].values[:, None] - lev[k]['mu_e_pade'].values[None, :]
                    wart = np.where(krok >= d, dp[:, None] + krok, -np.inf)
                    prev.append(wart.argmax(axis=0))
                    dp = wart.max(axis=0)
                j = int(dp.argmax())
                if np.isfinite(dp[j]) and (best is None or dp[j] > best[0]):
                    idx = [j]
                    for k in range(L - 2, -1, -1):
                        idx.append(int(prev[k][idx[-1]]))
                    idx = idx[::-1]
                    best = (float(dp[j]), [lev[k].iloc[idx[k]] for k in range(L)])
            if best is not None:
                return best[1][0], best[1][-1], d, best[1]
    return None


if KLATKI_ANOMALII is None:
    kl = pd.read_csv(PATH_CSV + 'result_lepkosc_anomalie_klatki.csv')
    klatki = kl.sort_values('udzial_dyskordantnych', ascending=False)['i'].head(LICZBA_KLATEK).astype(int).tolist()
else:
    klatki = list(KLATKI_ANOMALII)
print(f"[ANOM7] klatki: {klatki}")

sx, sy = x_d / 2, y_d / 2      # mapa we współrzędnych środków okien (jak w komórce 3d)
wiersze = []
for i in klatki:
    f_okna = PATH_CSV + f'real{i}_lepkosc_okna.csv'
    if not os.path.isfile(f_okna):
        print(f"[ANOM7] i={i}: brak {f_okna}")
        continue
    tab = pd.read_csv(f_okna)
    wynik = znajdz_skrajne(tab)
    if wynik is None:
        print(f"[ANOM7] i={i}: nie znaleziono łańcucha okien (f rośnie, mu_e/mu maleje)")
        continue
    A, B, prog, lancuch = wynik

    # tło: pole N jak w real{i}_2D_N_interp_lepkosc.png (z real{i}.csv; awaryjnie z real{i}_lepkosc_okna.csv)
    df = _wczytaj_csv(PATH_CSV + f'real{i}.csv')
    if df is not None:
        x, y, z, _ = _pelna_siatka_N(df)
    else:
        x, y, z = tab['x0'].to_numpy(float), tab['y0'].to_numpy(float), tab['N'].to_numpy(float)
    x = x + sx
    y = y + sy
    Xg, Yg, Z_int = _interpoluj_N(x, y, z)

    fig = _rysuj_2d_z_siatki(x, y, Xg, Yg, Z_int, CONC_LABEL,
                             f"{CONC_LABEL}, interpolacja ({INTERP_METHOD}) – klatka {i}\n"
                             f"okna A i B: f rośnie, mu_e/mu maleje")
    ax = fig.axes[0]
    ax.set_xlim(x.min() - sx, x.max() + sx)
    ax.set_ylim(y.min() - sy, y.max() + sy)
    ax.set_xlabel('x [px] (środek okna)')
    ax.set_ylabel('y [px] (środek okna)')
    for nazwa, w in (('A', A), ('B', B)):
        ax.add_patch(Rectangle((w['x0'], w['y0']), x_d, y_d, fill=False, edgecolor='white', linewidth=4.0, zorder=5))
        ax.add_patch(Rectangle((w['x0'], w['y0']), x_d, y_d, fill=False, edgecolor='black', linewidth=1.8, zorder=6))
        et = (f"{nazwa}\n"
              f"x0={int(w['x0'])}, y0={int(w['y0'])}\n"
              f"N={int(w['N'])}, f={_pl(w['f'], 4)}\n"
              f"Re_A3={_pl(w['Re_A3'], 3)}\n"
              f"μₑ/μ={_pl(w['mu_e_pade'], 4)}")
        ax.annotate(et, (w['x0'] + sx, w['y0'] + sy), ha='center', va='center', fontsize=8, zorder=7,
                    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='black', alpha=0.9))
        wiersze.append(dict(i=i, okno=nazwa, x0=int(w['x0']), y0=int(w['y0']), N=int(w['N']), f=w['f'],
                            Re_A3=w['Re_A3'], mu_e_przez_mu=w['mu_e_pade'], prog_kroku=prog, dlugosc_lancucha=len(lancuch)))
    fname = PATH_PNG + f'real{i}_2D_N_interp_anomalia.png'
    fig.savefig(fname, dpi=100)
    plt.close(fig)

    # 3D interpolacji N — tylko dla klatek z anomaliami
    fig3 = _rysuj_3d_z_siatki(x, y, z, Xg, Yg, Z_int, CONC_LABEL,
                              f"{CONC_LABEL}, interpolacja ({INTERP_METHOD}) – klatka {i}")
    fig3.savefig(PATH_PNG + f'real{i}_3D_N_interp.png', dpi=100)
    plt.close(fig3)
    print(f"[ANOM7] Zapisano: {fname}   A=({int(A['x0'])}; {int(A['y0'])}; {int(A['N'])}; {A['f']:.4f}; {A['Re_A3']:.3f}; {A['mu_e_pade']:.4f})"
          f"  B=({int(B['x0'])}; {int(B['y0'])}; {int(B['N'])}; {B['f']:.4f}; {B['Re_A3']:.3f}; {B['mu_e_pade']:.4f})")

if wiersze:
    out7 = pd.DataFrame(wiersze)
    out7.to_csv(PATH_CSV + 'result_lepkosc_anomalie_skrajne.csv', index=False, float_format='%.6g')
    print(f"[ANOM7] Zapisano: {PATH_CSV}result_lepkosc_anomalie_skrajne.csv")

# =====================================================================

In [ ]:
# =====================================================================
# Komórka 8: losowe bakterie w oknie 100x100 px — przypadki C (N=39) i D (N=43)
# =====================================================================
# Ten sam tor obliczeń co dla okien z rzeczywistych klatek (komórki 3b/3d):
#   punkty (px) -> /x_d, /y_d -> a = x + i*y w kwadracie jednostkowym -> lepkosc_okna(a, xp)
#   (f = N*pi*(R_BAKT/x_d)^2, sumy e33, e4, Re A3, mu_e/mu = (1+s)/(1-s), s = f + A3 f^3).
# Losowanie jest zgodne z warunkami, które spełniają punkty po czyszczeniu w prepare():
#   - odległość torusowa każdych dwóch środków >= r_min (usun_duplikaty_torus),
#   - każdy środek w osobnej komórce siatki DIST_SCALANIE x DIST_SCALANIE px (kwantyzuj_do_siatki
#     nie scala wtedy punktów i nie zmienia ich położeń, więc N pozostaje dokładnie równe zadanemu).
# Wklej PO komórce 3d (funkcja lepkosc_okna) — może być przed lub po pętli głównej (komórka 4).

PRZYPADKI_8   = {'C': 39, 'D': 43}   # liczby losowych bakterii w oknie
SEED_8        = 1                    # ziarno generatora (zmień, aby uzyskać inną realizację)
MAX_PROB_8    = 200_000              # limit prób losowania pojedynczego punktu
ZAPISZ_8      = True                 # zapis CSV i PNG do PATH_CSV
NORMUJ_OSIE_8 = True                 # True: osie x, y w [0, 1] (jak Figure 1); False: w pikselach

try:
    xp8 = cp                         # GPU (jak w prepare()), jeśli CuPy jest zaimportowane
except NameError:
    xp8 = np


def losuj_punkty_okna(N, rng, y_d=y_d, x_d=x_d, r_min=r_min, krok=DIST_SCALANIE):
    """N punktów jednostajnie w oknie x_d x y_d [px] (torus), spełniających warunki czyszczenia."""
    pts = np.empty((0, 2))
    komorki = set()
    proby = 0
    while len(pts) < N:
        proby += 1
        if proby > MAX_PROB_8:
            raise RuntimeError(f"Nie udało się wylosować {N} punktów (po {MAX_PROB_8} próbach).")
        p = np.array([rng.uniform(0, y_d), rng.uniform(0, x_d)])          # (y, x) jak w prepare()
        kom = (int(p[0] // krok), int(p[1] // krok))
        if kom in komorki:
            continue
        if len(pts):
            d = np.abs(pts - p)
            d = np.minimum(d, np.array([y_d, x_d]) - d)                   # odległość torusowa
            if np.min(np.hypot(d[:, 0], d[:, 1])) < r_min:
                continue
        pts = np.vstack([pts, p])
        komorki.add(kom)
    return pts


rng8 = np.random.default_rng(SEED_8)
wiersze8, punkty8 = [], {}
for nazwa, N in PRZYPADKI_8.items():
    pts = losuj_punkty_okna(N, rng8)                                      # (y, x) w pikselach
    punkty8[nazwa] = pts
    y_norm, x_norm = pts[:, 0] / y_d, pts[:, 1] / x_d                     # normalizacja do [0, 1]
    a = xp8.asarray(x_norm + 1j * y_norm, dtype=xp8.complex128)           # a = x + i*y
    lep = lepkosc_okna(a, xp8)
    wiersze8.append(dict(przypadek=nazwa, **lep))
    print(f"[{nazwa}] N={lep['N']}:  f = {lep['f']:.6f},  Re A3 = {lep['Re_A3']:.6f},  "
          f"mu_e/mu = {lep['mu_e_pade']:.6f}   (e33 = {lep['e33']:.6f}, Re e4 = {lep['Re_e4']:.6f})")

tab8 = pd.DataFrame(wiersze8)
print(tab8[['przypadek', 'N', 'f', 'Re_A3', 'mu_e_pade']].to_string(index=False, float_format=lambda v: f"{v:.6f}"))

if ZAPISZ_8:
    tab8.to_csv(PATH_CSV + 'result_losowe_C_D.csv', index=False, float_format='%.8g')
    # skala rysunku: okno jednostkowe [0,1]x[0,1] (współrzędne /x_d, /y_d) albo piksele
    Lx, Ly = (1.0, 1.0) if NORMUJ_OSIE_8 else (x_d, y_d)
    sk_x, sk_y = Lx / x_d, Ly / y_d                      # piksele -> jednostki osi
    rr = R_BAKT * sk_x                                   # promień dysku w jednostkach osi (3.5/100 = 0.035)
    fig8, osie = plt.subplots(1, 2, figsize=(9, 4.6))
    for ax, (nazwa, pts), w in zip(osie, punkty8.items(), wiersze8):
        kadr = Rectangle((0, 0), Lx, Ly, transform=ax.transData)
        for yy, xx in pts:
            for sx in (-Lx, 0, Lx):
                for sy in (-Ly, 0, Ly):
                    X, Y = xx * sk_x + sx, yy * sk_y + sy
                    if -rr < X < Lx + rr and -rr < Y < Ly + rr:
                        c_ = plt.Circle((X, Y), rr, facecolor='0.35', edgecolor='black', lw=0.4)
                        ax.add_patch(c_)
                        c_.set_clip_path(kadr)
        ax.add_patch(Rectangle((0, 0), Lx, Ly, fill=False, edgecolor='black', lw=0.8))
        m = 0.03 * Lx
        ax.set_xlim(-m, Lx + m); ax.set_ylim(-m, Ly + m); ax.set_aspect('equal')
        if NORMUJ_OSIE_8:
            ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
            ax.set_xlabel('x'); ax.set_ylabel('y')
        else:
            ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')
        ax.set_title(f"{nazwa}: N={w['N']}, f={w['f']:.4f}, Re A3={w['Re_A3']:.3f}, "
                     f"$\\mu_e/\\mu$={w['mu_e_pade']:.4f}", fontsize=9)
    fig8.tight_layout()
    fig8.savefig(PATH_PNG + 'real_losowe_C_D.png', dpi=200)
    plt.show()
    print(f"[K8] Zapisano: {PATH_CSV}result_losowe_C_D.csv oraz {PATH_PNG}real_losowe_C_D.png")
